# Headline

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Download

In [ ]:
!pip install -U --extra-index-url https://vnstocks.com/api/simple vnstock

Looking in indexes: https://pypi.org/simple, https://vnstocks.com/api/simple
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 287.0/287.0 kB 80.0 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.3/62.3 kB 173.9 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.0/75.0 kB 3.2 MB/s eta 0:00:00


## Import

In [ ]:
# 1. Thư viện chuẩn & Cấu hình cảnh báo
import math
import os
import pickle
import time
import warnings
from abc import ABC, abstractmethod
from collections import defaultdict

from statsmodels.tools.sm_exceptions import InterpolationWarning

warnings.filterwarnings("ignore")
warnings.simplefilter("ignore", InterpolationWarning)

# 2. Xử lý & Trực quan hóa dữ liệu, Lấy dữ liệu tài chính
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import vnstock
import xgboost as xgb
import yfinance as yf

# 3. Khoa học & Xử lý tín hiệu (SciPy)
from scipy import stats
from scipy.cluster.hierarchy import fcluster, linkage
from scipy.fft import rfft, rfftfreq
from scipy.spatial.distance import squareform

# 4. Thống kê & Chuỗi thời gian (Statsmodels)
from statsmodels.stats.diagnostic import acorr_ljungbox, het_arch
from statsmodels.stats.stattools import durbin_watson
from statsmodels.tsa.ar_model import AutoReg
from statsmodels.tsa.regime_switching.markov_regression import MarkovRegression
from statsmodels.tsa.stattools import acf, adfuller, bds, grangercausalitytests, kpss, pacf

# 5. Machine Learning & Đánh giá mô hình (Scikit-Learn)
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.ensemble import RandomForestRegressor
from sklearn.feature_selection import mutual_info_classif, mutual_info_regression
from sklearn.impute import SimpleImputer
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    brier_score_loss,
    classification_report,
    confusion_matrix,
    f1_score,
    log_loss,
    mean_squared_error,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.mixture import GaussianMixture
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.utils.class_weight import compute_sample_weight

# Data Input

In [ ]:
# =====================================================================
# BƯỚC: DATA PREPARATION (MULTI-ASSET DATA LOADER & CLEANING)
# =====================================================================
class MultiAssetDataPreparer:
    """
    Tải, kiểm toán hình học, chuẩn hóa giá điều chỉnh và lưu trữ dữ liệu
    cho toàn bộ rổ cổ phiếu trong final_portfolio.csv cùng chỉ số VN-Index.
    """
    def __init__(self, portfolio_path: str = 'final_portfolio.csv',
                 start_date: str = '2018-01-01', end_date: str = '2026-06-01',
                 source: str = 'KBS', cache_dir: str = 'data_cache'):
        self.portfolio_path = portfolio_path
        self.start_date = start_date
        self.end_date = end_date
        self.source = source
        self.cache_dir = cache_dir
        os.makedirs(self.cache_dir, exist_ok=True)

        # Đọc danh mục mục tiêu và trích xuất danh sách mã
        self.df_portfolio = pd.read_csv(self.portfolio_path)
        self.tickers = sorted(self.df_portfolio['Ticker'].unique().tolist())
        print(f"[*] Khởi tạo danh mục: {len(self.tickers)} mã cổ phiếu từ {self.portfolio_path}")

    def _fetch_history(self, symbol: str) -> pd.DataFrame:
        """Tải dữ liệu OHLCV từ Yahoo Finance để tránh bị chặn IP trên Colab."""
        cache_file = os.path.join(self.cache_dir, f"{symbol}.csv")

        if os.path.exists(cache_file):
            df = pd.read_csv(cache_file)
            df['time'] = pd.to_datetime(df['time'])
            df.set_index('time', inplace=True)
            return df

        try:
            # Chứng khoán VN trên Yahoo cần thêm hậu tố .VN (VD: VNINDEX.QA cho VN-Index, FPT.VN cho FPT)
            yf_symbol = "VNINDEX.QA" if symbol == "VNINDEX" else f"{symbol}.VN"

            ticker_obj = yf.Ticker(yf_symbol)
            df = ticker_obj.history(start=self.start_date, end=self.end_date, interval="1d")

            if df is not None and not df.empty:
                df.reset_index(inplace=True)

                # Đồng bộ tên cột với chuẩn code hiện tại của bạn
                df.rename(columns={
                    'Date': 'time', 'Open': 'open', 'High': 'high',
                    'Low': 'low', 'Close': 'close', 'Volume': 'volume'
                }, inplace=True)

                # Chuẩn hóa time (loại bỏ timezone nếu có)
                df['time'] = pd.to_datetime(df['time']).dt.tz_localize(None)
                df.sort_values(by='time', inplace=True)
                df.set_index('time', inplace=True)

                # Lưu file cache
                df.to_csv(cache_file)
                return df

        except Exception as err:
            print(f"  [!] Lỗi lấy dữ liệu {symbol} từ Yahoo Finance: {err}")

        return pd.DataFrame()

    def _clean_and_adjust(self, df: pd.DataFrame, df_mkt: pd.DataFrame) -> pd.DataFrame:
        """Kiểm toán hình học nến và bổ sung biến lợi nhuận thị trường."""
        df = df.copy()

        # 1. Khử trùng lặp và sắp xếp thứ tự thời gian
        df = df[~df.index.duplicated(keep='last')].sort_index()

        # 2. Kiểm toán biên hình học nến
        max_oc = df[['open', 'close']].max(axis=1)
        min_oc = df[['open', 'close']].min(axis=1)
        df['high'] = np.maximum(df['high'], max_oc)
        df['low'] = np.minimum(df['low'], min_oc)
        df = df[(df['low'] > 0) & (df['volume'] >= 0)]

        # 3. Chuẩn hóa giá điều chỉnh (nếu chưa có sẵn)
        if 'close_adj' not in df.columns:
            df['close_adj'] = df['close']
            ratio = df['close_adj'] / df['close']
            df['open_adj'] = df['open'] * ratio
            df['high_adj'] = df['high'] * ratio
            df['low_adj'] = df['low'] * ratio

        # 4. Tính các chuỗi Log Return cơ sở
        df['log_return'] = np.log(df['close_adj'] / df['close_adj'].shift(1))
        df['overnight_return'] = np.log(df['open_adj'] / df['close_adj'].shift(1))
        df['intraday_return'] = np.log(df['close_adj'] / df['open_adj'])
        df['hl_log_range'] = np.log(df['high_adj'] / df['low_adj'])

        # 5. Ghép lợi nhuận VN-Index làm biến ngoại sinh
        if not df_mkt.empty:
            df = df.join(df_mkt, how='left')
            df[df_mkt.columns] = df[df_mkt.columns].fillna(0.0)

        return df.dropna(subset=['close_adj', 'open_adj'])

    def prepare_dataset(self) -> dict[str, pd.DataFrame]:
        """Thực thi tải và làm sạch toàn bộ dữ liệu danh mục."""
        print("[*] Đang tải dữ liệu Benchmark VN-INDEX...")
        df_mkt = self._fetch_history('VNINDEX')
        if not df_mkt.empty:

            df_mkt.index = pd.to_datetime(df_mkt.index)
            df_mkt.sort_index(inplace=True)

            # Trích xuất các biến cơ bản của Market Benchmark
            df_mkt = df_mkt[['close', 'volume', 'high', 'low']].rename(columns={
                'close': 'mkt_close',
                'volume': 'mkt_volume',
                'high': 'mkt_high',
                'low': 'mkt_low'
            })
            df_mkt['mkt_return'] = np.log(df_mkt['mkt_close'] / df_mkt['mkt_close'].shift(1))


        clean_dict = {}
        for idx, ticker in enumerate(self.tickers, 1):
            print(f"[{idx:02d}/{len(self.tickers):02d}] Chuẩn bị dữ liệu: {ticker}...")
            df_raw = self._fetch_history(ticker)
            if not df_raw.empty and len(df_raw) > 50:
                clean_df = self._clean_and_adjust(df_raw, df_mkt)
                clean_dict[ticker] = clean_df
            else:
                print(f"  [!] Bỏ qua {ticker} (dữ liệu rỗng hoặc không đủ số phiên tối thiểu).")
            time.sleep(5)
        print(f"\n[✓] HOÀN TẤT BƯỚC DATA PREP: Sẵn sàng {len(clean_dict)}/{len(self.tickers)} mã cổ phiếu cho khâu Modeling.")
        return clean_dict

## Port. Index Preparation:

In [ ]:
# =====================================================================
# THỰC THI CHUẨN BỊ DỮ LIỆU
# =====================================================================
portfolio_path = "/content/drive/MyDrive/Colab Notebooks/AI agent in Portfolio Management/Compiled Dataset/final_portfolio.csv"

data_preparer = MultiAssetDataPreparer(
    portfolio_path=portfolio_path,
    start_date='2018-01-01',
    end_date='2026-06-01',
    source='KBS'
)

clean_price_dict = data_preparer.prepare_dataset()

[*] Khởi tạo danh mục: 49 mã cổ phiếu từ /content/drive/MyDrive/Colab Notebooks/AI agent in Portfolio Management/Compiled Dataset/final_portfolio.csv
[*] Đang tải dữ liệu Benchmark VN-INDEX...


ERROR:yfinance:HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: VNINDEX.QA"}}}
ERROR:yfinance:$VNINDEX.QA: possibly delisted; no timezone found


[01/49] Chuẩn bị dữ liệu: ABT...
[02/49] Chuẩn bị dữ liệu: ADP...
[03/49] Chuẩn bị dữ liệu: BMP...
[04/49] Chuẩn bị dữ liệu: BRC...
[05/49] Chuẩn bị dữ liệu: CHP...
[06/49] Chuẩn bị dữ liệu: CSM...
[07/49] Chuẩn bị dữ liệu: CSV...
[08/49] Chuẩn bị dữ liệu: CTI...
[09/49] Chuẩn bị dữ liệu: DGC...
[10/49] Chuẩn bị dữ liệu: DHC...
[11/49] Chuẩn bị dữ liệu: DPR...
[12/49] Chuẩn bị dữ liệu: DRL...
[13/49] Chuẩn bị dữ liệu: DSN...
[14/49] Chuẩn bị dữ liệu: DVP...
[15/49] Chuẩn bị dữ liệu: FMC...
[16/49] Chuẩn bị dữ liệu: FPT...
[17/49] Chuẩn bị dữ liệu: GHC...
[18/49] Chuẩn bị dữ liệu: HDG...
[19/49] Chuẩn bị dữ liệu: HT1...
[20/49] Chuẩn bị dữ liệu: HTG...
[21/49] Chuẩn bị dữ liệu: HTI...
[22/49] Chuẩn bị dữ liệu: LGC...
[23/49] Chuẩn bị dữ liệu: LIX...
[24/49] Chuẩn bị dữ liệu: MCH...
[25/49] Chuẩn bị dữ liệu: MWG...
[26/49] Chuẩn bị dữ liệu: NCT...
[27/49] Chuẩn bị dữ liệu: NNC...
[28/49] Chuẩn bị dữ liệu: NT2...
[29/49] Chuẩn bị dữ liệu: PTB...
[30/49] Chuẩn bị dữ liệu: REE...
[31/49] Ch

## Data Visualization

In [ ]:
print(len(clean_price_dict.keys()))
print(clean_price_dict.keys())

49
dict_keys(['ABT', 'ADP', 'BMP', 'BRC', 'CHP', 'CSM', 'CSV', 'CTI', 'DGC', 'DHC', 'DPR', 'DRL', 'DSN', 'DVP', 'FMC', 'FPT', 'GHC', 'HDG', 'HT1', 'HTG', 'HTI', 'LGC', 'LIX', 'MCH', 'MWG', 'NCT', 'NNC', 'NT2', 'PTB', 'REE', 'SAB', 'SAV', 'SBA', 'SCS', 'SHP', 'SJD', 'SMB', 'TCL', 'TCM', 'TDM', 'TLG', 'TRA', 'TRC', 'TTA', 'VIP', 'VMD', 'VNM', 'VPD', 'VTO'])


In [ ]:
# =====================================================================
# 5. HIỂN THỊ KIỂM TOÁN VẬT LÝ VÀ CHẨN ĐOÁN NANs ĐẦU VÀO
# =====================================================================
def raw_data_profile(df_raw, SYMBOL):
    print("\n" + "=" * 80)
    print(f"TỔNG QUAN DATASET KẾT HỢP (DAILY + MICROSTRUCTURE):")
    print(f"- Số dòng tổng cộng      : {len(df_raw):,}")
    print(f"- Mã cổ phiếu            : {SYMBOL}")
    print(f"- Khung thời gian        : Từ {df_raw.index.min().date()} đến {df_raw.index.max().date()}")
    print(f"- Tổng giá trị khuyết NaN: {df_raw.isna().sum().sum()}")
    print("=" * 80)

    # Hiển thị danh sách các cột hiện có để xác nhận Vi cấu trúc đã được gộp thành công
    print(f"\n[Các cột dữ liệu hiện có]:\n{df_raw.columns.tolist()}\n")

    # Hiển thị cấu trúc mẫu
    display(df_raw.head())
    display(df_raw.tail())

    print("\n" * 2)
    print(f"No. NaN values: {df_raw.isna().sum().sum()}")
    if df_raw.isna().sum().sum() > 0:
        display(df_raw.isna().sum()[df_raw.isna().sum() > 0])
    else:
        print("Dữ liệu hoàn hảo, không có giá trị khuyết thiếu!")


for key, value in clean_price_dict.items():
    raw_data_profile(df_raw=value, SYMBOL=key)


TỔNG QUAN DATASET KẾT HỢP (DAILY + MICROSTRUCTURE):
- Số dòng tổng cộng      : 2,186
- Mã cổ phiếu            : ABT
- Khung thời gian        : Từ 2018-01-01 đến 2026-05-29
- Tổng giá trị khuyết NaN: 2

[Các cột dữ liệu hiện có]:
['open', 'high', 'low', 'close', 'volume', 'Dividends', 'Stock Splits', 'close_adj', 'open_adj', 'high_adj', 'low_adj', 'log_return', 'overnight_return', 'intraday_return', 'hl_log_range']



,open,high,low,close,volume,Dividends,Stock Splits,close_adj,open_adj,high_adj,low_adj,log_return,overnight_return,intraday_return,hl_log_range
time,,,,,,,,,,,,,,,
2018-01-01,17450.095703,17450.095703,17450.095703,17450.095703,0,0.0,0.0,17450.095703,17450.095703,17450.095703,17450.095703,NaN,NaN,0.000000,0.000000
2018-01-02,18031.765625,18031.765625,18031.765625,18031.765625,190,0.0,0.0,18031.765625,18031.765625,18031.765625,18031.765625,0.032790,0.032790,0.000000,0.000000
2018-01-03,16781.174511,18031.764639,16781.174511,17973.597656,1480,0.0,0.0,17973.597656,16781.174511,18031.764639,16781.174511,-0.003231,-0.071877,0.068646,0.071877
2018-01-04,17450.093006,18613.432540,16781.172774,18322.597656,780,0.0,0.0,18322.597656,17450.093006,18613.432540,16781.172774,0.019231,-0.029559,0.048790,0.103626
2018-01-05,17973.597694,18322.599591,17101.092951,18206.265625,320,0.0,0.0,18206.265625,17973.597694,18322.599591,17101.092951,-0.006369,-0.019231,0.012862,0.068993


,open,high,low,close,volume,Dividends,Stock Splits,close_adj,open_adj,high_adj,low_adj,log_return,overnight_return,intraday_return,hl_log_range
time,,,,,,,,,,,,,,,
2026-05-25,55200.0,55200.0,55200.0,55200.0,1700,0.0,0.0,55200.0,55200.0,55200.0,55200.0,0.001813,0.001813,0.000000,0.000000
2026-05-26,55900.0,56500.0,55900.0,56100.0,7000,0.0,0.0,56100.0,55900.0,56500.0,55900.0,0.016173,0.012601,0.003571,0.010676
2026-05-27,56100.0,56400.0,56100.0,56400.0,2300,0.0,0.0,56400.0,56100.0,56400.0,56100.0,0.005333,0.000000,0.005333,0.005333
2026-05-28,56700.0,57000.0,56400.0,56400.0,4000,0.0,0.0,56400.0,56700.0,57000.0,56400.0,0.000000,0.005305,-0.005305,0.010582
2026-05-29,56000.0,56000.0,56000.0,56000.0,200,0.0,0.0,56000.0,56000.0,56000.0,56000.0,-0.007117,-0.007117,0.000000,0.000000





No. NaN values: 2


,0
log_return,1
overnight_return,1


...


TỔNG QUAN DATASET KẾT HỢP (DAILY + MICROSTRUCTURE):
- Số dòng tổng cộng      : 2,182
- Mã cổ phiếu            : VTO
- Khung thời gian        : Từ 2018-01-01 đến 2026-05-29
- Tổng giá trị khuyết NaN: 2

[Các cột dữ liệu hiện có]:
['open', 'high', 'low', 'close', 'volume', 'Dividends', 'Stock Splits', 'close_adj', 'open_adj', 'high_adj', 'low_adj', 'log_return', 'overnight_return', 'intraday_return', 'hl_log_range']



,open,high,low,close,volume,Dividends,Stock Splits,close_adj,open_adj,high_adj,low_adj,log_return,overnight_return,intraday_return,hl_log_range
time,,,,,,,,,,,,,,,
2018-01-01,3892.095215,3892.095215,3892.095215,3892.095215,0,0.0,0.0,3892.095215,3892.095215,3892.095215,3892.095215,NaN,NaN,0.000000,0.000000
2018-01-02,3923.231126,3923.231126,3780.891675,3807.580322,160950,0.0,0.0,3807.580322,3923.231126,3923.231126,3780.891675,-0.021954,7.967953e-03,-0.029922,0.036956
2018-01-03,3807.580383,3883.198218,3758.651196,3785.339844,153940,0.0,0.0,3785.339844,3807.580383,3883.198218,3758.651196,-0.005858,1.604874e-08,-0.005858,0.032599
2018-01-04,3758.651279,3869.853980,3758.651279,3860.957764,92420,0.0,0.0,3860.957764,3758.651279,3869.853980,3758.651279,0.019780,-7.075479e-03,0.026855,0.029157
2018-01-05,3834.269753,3860.958405,3780.892448,3825.373535,191030,0.0,0.0,3825.373535,3834.269753,3860.958405,3780.892448,-0.009259,-6.936278e-03,-0.002323,0.020955


,open,high,low,close,volume,Dividends,Stock Splits,close_adj,open_adj,high_adj,low_adj,log_return,overnight_return,intraday_return,hl_log_range
time,,,,,,,,,,,,,,,
2026-05-25,10781.557617,10872.541226,10781.557617,10781.557617,111900,0.0,0.0,10781.557617,10781.557617,10872.541226,10781.557617,0.000000,0.000000e+00,0.000000,0.008403
2026-05-26,10827.048625,11054.507629,10827.048625,10918.032227,178800,0.0,0.0,10918.032227,10827.048625,11054.507629,10827.048625,0.012579,4.210459e-03,0.008368,0.020791
2026-05-27,11100.000210,11100.000210,10918.032993,11009.016602,214300,0.0,0.0,11009.016602,11100.000210,11100.000210,10918.032993,0.008299,1.652937e-02,-0.008230,0.016529
2026-05-28,11009.016602,11054.508406,10963.524797,11009.016602,107800,0.0,0.0,11009.016602,11009.016602,11054.508406,10963.524797,0.000000,0.000000e+00,0.000000,0.008265
2026-05-29,11009.016393,11190.983607,11009.016393,11100.000000,248800,0.0,0.0,11100.000000,11009.016393,11190.983607,11009.016393,0.008230,-1.890449e-08,0.008230,0.016394





No. NaN values: 2


,0
log_return,1
overnight_return,1


### làm sạch là loại bỏ NaN trong Return:

In [ ]:
"""
for key, value in clean_price_dict.items():
    clean_price_dict[key] = value.dropna()
    raw_data_profile(df_raw=clean_price_dict[key], SYMBOL=key)
"""
# Lưu dict vào file nhị phân (.pkl)
with open("clean_price_dict.pkl", "wb") as f:
    pickle.dump(clean_price_dict, f)


In [ ]:
# Đọc lại dict từ file
with open("clean_price_dict.pkl", "rb") as f:
    loaded_dict = pickle.load(f)

# Kiểm tra dữ liệu
print(loaded_dict["VNM"].head().to_markdown())

| time                |    open |    high |     low |   close |           volume |   Dividends |   Stock Splits |   close_adj |   open_adj |   high_adj |   low_adj |   log_return |   overnight_return |   intraday_return |   hl_log_range |
|:--------------------|--------:|--------:|--------:|--------:|-----------------:|------------:|---------------:|------------:|-----------:|-----------:|----------:|-------------:|-------------------:|------------------:|---------------:|
| 2018-01-01 00:00:00 | 96142.4 | 96142.4 | 96142.4 | 96142.4 |      0           |           0 |              0 |     96142.4 |    96142.4 |    96142.4 |   96142.4 | nan          |       nan          |        0          |     0          |
| 2018-01-02 00:00:00 | 97248.5 | 97663.4 | 96142.4 | 97479   | 536774           |           0 |              0 |     97479   |    97248.5 |    97663.4 |   96142.4 |   0.0138064  |         0.0114396  |        0.00236684 |     0.015696   |
| 2018-01-03 00:00:00 | 98400.8 | 99092.1 | 

# Feature Engineering

## BƯỚC 1: XÁC LẬP BÀI TOÁN, KIỂM TOÁN VẬT LÝ VÀ CHUẨN HÓA TOPOLOGY DỮ LIỆU
Mục tiêu: Đảm bảo dữ liệu thô sạch hoàn toàn về mặt vật lý, xác lập không gian xác suất và tập thông tin $\mathcal{I}_t$.

In [ ]:
# =====================================================================
# BƯỚC 1: XÁC LẬP BÀI TOÁN & CHUẨN HÓA TOPOLOGY ĐƠN TÀI SẢN (SINGLE-ASSET)
# Tích hợp nền tảng L2 Order Book Đa tầng (Multi-Level) & Chống Leakage
# =====================================================================

class Layer1DataIntegrity:
    @staticmethod
    def run_audit(df: pd.DataFrame) -> tuple[pd.DataFrame, dict]:
        df = df.copy()
        initial_len = len(df)

        if df.index.duplicated().any():
            df = df[~df.index.duplicated(keep='last')]
        df = df.sort_index()

        max_oc = df[['open', 'close']].max(axis=1)
        min_oc = df[['open', 'close']].min(axis=1)
        df['high'] = np.maximum(df['high'], max_oc)
        df['low'] = np.minimum(df['low'], min_oc)
        df = df[(df['low'] > 0) & (df['volume'] >= 0)]

        if 'split_factor' not in df.columns:
            df['split_factor'] = 1.0
        if 'dividend' not in df.columns:
            df['dividend'] = 0.0

        # [ĐÃ SỬA LỖI LEAKAGE]: Synthetic Total Return Index
        # Mọi tính toán ở thời điểm t chỉ sử dụng close(t), dividend(t), split(t) và close(t-1)
        daily_ret = ((df['close'] + df['dividend']) / (df['close'].shift(1) * df['split_factor'])) - 1.0

        # Bắt đầu tại 1000.0, nhân dồn (cumprod) thuần tiến không chứa lookahead
        df['close_adj'] = 1000.0 * (1.0 + daily_ret.fillna(0)).cumprod()

        ratio = df['close_adj'] / df['close']
        for col in ['open', 'high', 'low']:
            df[f'{col}_adj'] = df[col] * ratio

        dropped_zero_var = []
        for col in df.columns:
            if col not in ['split_factor', 'dividend', 'target_label']:
                if pd.api.types.is_numeric_dtype(df[col]):
                    if df[col].std() < 1e-8 or df[col].nunique() <= 1:
                        dropped_zero_var.append(col)

        if dropped_zero_var:
            df.drop(columns=dropped_zero_var, inplace=True)

        return df, {
            "initial_bars": initial_len,
            "clean_bars": len(df),
            "dropped_flat_cols": dropped_zero_var
        }

class Layer2MicrostructureTopologies:
    """
    Khai phá Topology: Mở rộng không gian trạng thái với dữ liệu vi cấu trúc.
    Xử lý dữ liệu Sổ lệnh L2 Đa tầng (Multi-level Order Book) lên đến N bước giá.
    """
    @staticmethod
    def compute(df: pd.DataFrame, max_depth_levels: int = 10) -> pd.DataFrame:
        df = df.copy()

        # 1. Topo Tiêu chuẩn (OHLCV)
        df['log_return'] = np.log(df['close_adj'] / df['close_adj'].shift(1))
        df['overnight_return'] = np.log(df['open_adj'] / df['close_adj'].shift(1))
        df['intraday_return'] = np.log(df['close_adj'] / df['open_adj'])
        df['hl_log_range'] = np.log(df['high_adj'] / df['low_adj'])

        # 2. Topo Vi cấu trúc (L2 Multi-Level Order Book)
        # Tự động quét số lượng tầng (depth) khả dụng trong dataset
        available_levels = []
        for i in range(1, max_depth_levels + 1):
            if f'bid_price_{i}' in df.columns and f'ask_price_{i}' in df.columns:
                available_levels.append(i)

        # Trạng thái A: Có dữ liệu Đa tầng (Multi-level L2)
        if len(available_levels) > 1:
            df['mid_price'] = (df['bid_price_1'] + df['ask_price_1']) / 2.0
            df['l2_spread'] = df['ask_price_1'] - df['bid_price_1']

            total_weighted_bid_size = np.zeros(len(df))
            total_weighted_ask_size = np.zeros(len(df))
            cross_weighted_bid_price = np.zeros(len(df))
            cross_weighted_ask_price = np.zeros(len(df))

            # Quét qua các tầng sổ lệnh, áp dụng Exponential Decay Weight
            # để giảm thiểu nhiễu từ các lệnh ảo (Spoofing) đặt ở giá xa
            for i in available_levels:
                decay_weight = np.exp(-(i - 1) * 0.5)

                b_size_w = df[f'bid_size_{i}'] * decay_weight
                a_size_w = df[f'ask_size_{i}'] * decay_weight

                total_weighted_bid_size += b_size_w
                total_weighted_ask_size += a_size_w

                cross_weighted_bid_price += df[f'bid_price_{i}'] * b_size_w
                cross_weighted_ask_price += df[f'ask_price_{i}'] * a_size_w

            imbalance_denominator = total_weighted_bid_size + total_weighted_ask_size + 1e-8

            # Tính Depth Order Book Imbalance (OBI đa tầng)
            df['order_imbalance_depth'] = (total_weighted_bid_size - total_weighted_ask_size) / imbalance_denominator

            # Tính Depth Micro-Price (Giao cắt chéo khối lượng 2 bên)
            # Thể hiện điểm cân bằng lực cung cầu thực sự của toàn bộ sổ lệnh
            df['micro_price_depth'] = (
                (cross_weighted_bid_price / (total_weighted_bid_size + 1e-8)) * total_weighted_ask_size +
                (cross_weighted_ask_price / (total_weighted_ask_size + 1e-8)) * total_weighted_bid_size
            ) / imbalance_denominator

            # Độ lệch định hướng vi mô sắc bén nhất
            df['micro_mid_deviation'] = df['micro_price_depth'] - df['mid_price']

        # Trạng thái B: Tương thích ngược - Chỉ có Top-of-book L1 (best_bid, best_ask)
        elif all(col in df.columns for col in ['best_bid', 'best_ask', 'bid_size', 'ask_size']):
            df['mid_price'] = (df['best_bid'] + df['best_ask']) / 2.0
            imbalance_denominator = df['bid_size'] + df['ask_size'] + 1e-8

            df['micro_price'] = (df['best_bid'] * df['ask_size'] + df['best_ask'] * df['bid_size']) / imbalance_denominator
            df['l2_spread'] = df['best_ask'] - df['best_bid']
            df['micro_mid_deviation'] = df['micro_price'] - df['mid_price']
            df['order_imbalance_depth'] = (df['bid_size'] - df['ask_size']) / imbalance_denominator

        return df


class Layer0TripleBarrier:
    """
    LONG-ONLY ASYMMETRIC META-LABELING:
    - Primary Model giả định: Tín hiệu Long (Mua).
    - Target nhị phân:
        1: Chạm Upper Barrier (Take Profit) trước.
        0: Chạm Lower Barrier (Stop Loss) trước hoặc Time-out không đạt target.
    """
    @staticmethod
    def label(df: pd.DataFrame, h: int = 5, pt: float = 1.0, sl: float = 1.0, vol_span: int = 20) -> pd.DataFrame:
        df = df.copy()
        log_ret = df['log_return']

        # Biến động cục bộ dùng để co giãn động độ rộng của rào cản
        sigma = log_ret.ewm(span=vol_span, min_periods=2).std().bfill()

        prices = df['close_adj'].values
        sigmas = sigma.values
        n = len(df)

        meta_target = np.full(n, np.nan)
        ret_barrier = np.full(n, np.nan)
        touch_time = np.full(n, np.nan, dtype=object)

        for i in range(n - h):
            p0 = prices[i]

            if pd.isna(p0) or pd.isna(sigmas[i]):
                continue

            # Rào trên và rào dưới theo độ biến động cục bộ
            upper_barrier = p0 * (1.0 + pt * sigmas[i])
            lower_barrier = p0 * (1.0 - sl * sigmas[i])

            window_prices = prices[i + 1 : i + h + 1]
            hit_upper = np.where(window_prices >= upper_barrier)[0]
            hit_lower = np.where(window_prices <= lower_barrier)[0]

            first_upper = hit_upper[0] if len(hit_upper) > 0 else np.inf
            first_lower = hit_lower[0] if len(hit_lower) > 0 else np.inf

            # Logic nhãn Long-Only Binary Meta-Label
            if first_upper < first_lower and first_upper < np.inf:
                meta_target[i] = 1.0  # Lệnh Long thắng (Take-Profit thành công)
                ret_barrier[i] = (window_prices[first_upper] / p0) - 1.0
                touch_time[i] = df.index[i + 1 + first_upper]
            else:
                meta_target[i] = 0.0  # Lệnh Long thua (Chạm Cut-Loss hoặc Sideway hết h phiên)
                if first_lower < first_upper and first_lower < np.inf:
                    ret_barrier[i] = (window_prices[first_lower] / p0) - 1.0
                    touch_time[i] = df.index[i + 1 + first_lower]
                else:
                    ret_barrier[i] = (window_prices[-1] / p0) - 1.0
                    touch_time[i] = df.index[i + h]

        df['local_volatility'] = sigma
        df['target_label'] = meta_target              # Target phân loại nhị phân {0, 1}
        df['target_barrier_return'] = ret_barrier     # Return thực tế tại điểm chạm rào
        df['barrier_touch_time'] = touch_time
        return df

## BƯỚC 2: CHẨN ĐOÁN TOÀN DIỆN ĐẶC TÍNH QUÁ TRÌNH SINH DỮ LIỆU (DGP DIAGNOSTICS)
Mục tiêu: Sử dụng các kiểm định thống kê và kinh tế lượng vi cấu trúc trên mảng dữ liệu đầu vào (chưa dùng target) để thu thập bằng chứng thực nghiệm.

In [ ]:
# =====================================================================
# CÁC LỚP CHẨN ĐOÁN (Single-Asset Data-Driven Diagnostics)
# =====================================================================

class Layer3DistributionalDiagnostics:
    @staticmethod
    def run(returns: pd.Series) -> dict:
        r = returns.dropna()
        if len(r) < 30:
            return {"is_leptokurtic": False, "reject_normality": False}

        skew = float(stats.skew(r, bias=False))
        kurt = float(stats.kurtosis(r, fisher=False, bias=False))
        jb_stat, jb_p = stats.jarque_bera(r)

        return {
            "mean": float(np.mean(r)),
            "std": float(np.std(r, ddof=1)),
            "skewness": skew,
            "kurtosis": kurt,
            "is_leptokurtic": bool(kurt > 3.0),
            "jarque_bera_stat": float(jb_stat),
            "jarque_bera_pvalue": float(jb_p),
            "reject_normality": bool(jb_p < 0.05)
        }

class Layer4MemoryDependence:
    @staticmethod
    def compute_hurst_dfa(series: pd.Series) -> float:
        s = series.dropna().values
        y = np.cumsum(s - np.mean(s))
        n = len(y)
        if n < 40:
            return 0.5

        scales = np.floor(np.logspace(np.log10(10), np.log10(n // 4), num=15)).astype(int)
        scales = np.unique(scales)
        fluctuations = []

        for s_len in scales:
            num_segments = n // s_len
            segment_fluct = []
            for i in range(num_segments):
                seg = y[i * s_len : (i + 1) * s_len]
                x = np.arange(s_len)
                poly = np.polyfit(x, seg, 1)
                trend = np.polyval(poly, x)
                segment_fluct.append(np.sqrt(np.mean((seg - trend) ** 2)))
            fluctuations.append(np.mean(segment_fluct))

        poly_hurst = np.polyfit(np.log(scales), np.log(fluctuations), 1)
        return float(poly_hurst[0])

    @staticmethod
    def lo_mackinlay_vr(prices: pd.Series, k: int) -> dict:
        p = np.log(prices.dropna().values)
        t = len(p)
        if t <= k + 2:
            return {"vr": 1.0, "z_stat": 0.0, "p_value": 1.0}

        r1 = p[1:] - p[:-1]
        mu = (p[-1] - p[0]) / (t - 1)
        var_1 = np.sum((r1 - mu) ** 2) / (t - 2)

        rk = p[k:] - p[:-k]
        m = k * (t - k) * (1 - (k / (t - 1)))
        var_k = np.sum((rk - k * mu) ** 2) / m
        vr = var_k / var_1

        delta = np.zeros(k - 1)
        denom = (np.sum((r1 - mu) ** 2)) ** 2
        for j in range(1, k):
            num = np.sum(((r1[j:] - mu) ** 2) * ((r1[:-j] - mu) ** 2))
            delta[j - 1] = ((2.0 * (k - j) / k) ** 2) * (num / denom)

        phi_k = np.sum(delta)
        z_star = (vr - 1.0) / np.sqrt(phi_k) if phi_k > 0 else 0.0
        p_val = 2.0 * (1.0 - stats.norm.cdf(abs(z_star)))

        return {"vr": float(vr), "z_stat": float(z_star), "p_value": float(p_val)}

    @classmethod
    def multi_scale_memory(cls, prices: pd.Series, returns: pd.Series) -> dict:
        r = returns.dropna()
        if len(r) < 50:
            return {
                "dynamic_lags": [1, 3, 5], "hurst": 0.5,
                "short_term_mean_reverting": False, "long_term_trending": False
            }

        max_lag = min(30, len(r) // 3)
        acf_vals = acf(r, nlags=max_lag, fft=True)
        conf_interval = 1.96 / np.sqrt(len(r))
        sig_lags = [i for i, val in enumerate(acf_vals[1:], 1) if abs(val) > conf_interval]

        if not sig_lags:
            sig_lags = [1, 3, 5]

        vr_short = cls.lo_mackinlay_vr(prices, k=3)
        vr_long = cls.lo_mackinlay_vr(prices, k=20)

        return {
            "dynamic_lags": sorted(list(set(sig_lags[:5] + [1, 5, 20]))),
            "hurst": cls.compute_hurst_dfa(returns),
            "short_term_mean_reverting": bool(vr_short['vr'] < 1.0 and vr_short['p_value'] < 0.05),
            "long_term_trending": bool(vr_long['vr'] > 1.0 and vr_long['p_value'] < 0.05),
            "vr_short_stat": vr_short,
            "vr_long_stat": vr_long
        }

class Layer4bSpectralCycleDiagnostics:
    """Phát hiện bước sóng chu kỳ chi phối (Fast Fourier Transform)."""
    @staticmethod
    def run(prices: pd.Series) -> dict:
        p = prices.dropna().values
        n = len(p)
        if n < 40:
            return {"dominant_cycle_len": 10}

        detrended = p - np.polyval(np.polyfit(np.arange(n), p, 1), np.arange(n))
        fft_vals = np.abs(rfft(detrended))
        freqs = rfftfreq(n, d=1.0)

        fft_vals[0] = 0
        peak_idx = np.argmax(fft_vals)
        dominant_freq = freqs[peak_idx] if freqs[peak_idx] > 0 else 0.1
        dominant_cycle = int(np.clip(1.0 / dominant_freq, 3, 30))

        return {"dominant_cycle_len": dominant_cycle}

class Layer5FractionalIntegration:
    @staticmethod
    def get_weights(d: float, size: int, threshold: float = 1e-4) -> np.ndarray:
        w = [1.0]
        for k in range(1, size):
            w_k = -w[-1] / k * (d - k + 1)
            if abs(w_k) < threshold:
                break
            w.append(w_k)
        return np.array(w[::-1])

    @classmethod
    def fractionally_diff(cls, series: pd.Series, d: float, threshold: float = 1e-4) -> pd.Series:
        weights = cls.get_weights(d, len(series), threshold)
        width = len(weights)
        vals = series.values
        res = [np.dot(weights, vals[i - width : i]) for i in range(width, len(vals))]
        return pd.Series(res, index=series.index[width:], name=f"frac_diff_{d:.2f}")

    @classmethod
    def find_optimal_d(cls, series: pd.Series, d_step: float = 0.05) -> dict:
        s = series.dropna()
        if len(s) < 50:
            return {"optimal_d": 1.0, "price_stationary": False, "memory_retention_corr": 0.0}

        adf_raw_p = adfuller(s, autolag='AIC')[1]

        with warnings.catch_warnings():
            warnings.filterwarnings("ignore", category=InterpolationWarning)
            kpss_stat, kpss_raw_p, _, _ = kpss(s, regression='c', nlags="auto")

        best_d = 1.0
        best_corr = 0.0

        for d in np.arange(0.0, 1.05, d_step):
            fd = cls.fractionally_diff(s, d)
            if len(fd) < 50:
                continue
            adf_p = adfuller(fd.dropna(), autolag='AIC')[1]
            if adf_p < 0.05:
                best_d = float(round(d, 2))
                aligned = pd.concat([s, fd], axis=1).dropna()
                best_corr = float(np.corrcoef(aligned.iloc[:, 0], aligned.iloc[:, 1])[0, 1])
                break

        return {
            "price_stationary": bool(adf_raw_p < 0.05 and kpss_raw_p > 0.05),
            "optimal_d": best_d,
            "memory_retention_corr": best_corr
        }

class Layer6VolatilityDynamics:
    @staticmethod
    def estimate(df: pd.DataFrame) -> dict:
        c, o = df['close_adj'], df['open_adj']
        ret = np.log(c / c.shift(1)).dropna()
        if len(ret) < 30:
            return {"has_arch_effect": False, "has_asymmetric_vol": False, "leverage_effect_corr": 0.0}

        lm_stat, lm_p, _, _ = het_arch(ret, nlags=5)

        ret_lag = ret.shift(1).dropna()
        vol_proxy = (ret**2).iloc[1:]
        aligned_df = pd.concat([ret_lag, vol_proxy], axis=1).dropna()
        leverage_corr = aligned_df.iloc[:,0].corr(aligned_df.iloc[:,1])

        return {
            "arch_lm_stat": float(lm_stat),
            "arch_lm_pvalue": float(lm_p),
            "has_arch_effect": bool(lm_p < 0.05),
            "leverage_effect_corr": float(leverage_corr),
            "has_asymmetric_vol": bool(leverage_corr < -0.1)
        }

class Layer6bVolatilityJumpDiagnostics:
    """Kiểm định tách biến động bước nhảy (Barndorff-Nielsen & Shephard Jump Test)."""
    @staticmethod
    def run(df: pd.DataFrame, window: int = 20) -> dict:
        r = df['log_return'].dropna()
        if len(r) < window * 2:
            return {"has_volatility_jumps": False, "mean_jump_ratio": 0.0}

        rv = (r ** 2).rolling(window).sum()
        abs_r = r.abs()
        bv = (np.pi / 2.0) * (abs_r * abs_r.shift(1)).rolling(window).sum()

        jump_ratio = np.maximum(rv - bv, 0) / (rv + 1e-8)
        significant_jumps = (jump_ratio > 0.25).sum()

        return {
            "has_volatility_jumps": bool(significant_jumps > (len(r) * 0.05)),
            "mean_jump_ratio": float(jump_ratio.mean())
        }

class Layer7bVolumeDiagnostics:
    @staticmethod
    def run(df: pd.DataFrame) -> dict:
        v = df['volume'].dropna()
        r_abs = np.log(df['close_adj'] / df['close_adj'].shift(1)).abs().dropna()

        aligned = pd.concat([v, r_abs], axis=1).dropna()
        if aligned.empty or len(aligned) < 30:
            return {"is_volume_significant": False, "vol_ret_corr": 0.0}

        corr, p_val = stats.spearmanr(aligned.iloc[:, 0], aligned.iloc[:, 1])

        return {
            "vol_ret_corr": float(corr),
            "is_volume_significant": bool(p_val < 0.05 and corr > 0.1)
        }

class Layer7cMicrostructureDiagnostics:
    """
    Chẩn đoán độ nhạy và sức mạnh dự báo của dữ liệu Sổ lệnh (L2).
    Khai thác trực tiếp chênh lệch Micro-price và Mid-price từ Stage 1.
    """
    @staticmethod
    def run(df: pd.DataFrame) -> dict:
        r = df['log_return'].fillna(0)

        # Chỉ kích hoạt nếu Stage 1 phát hiện có dữ liệu Sổ lệnh (L2)
        if 'micro_mid_deviation' not in df.columns or 'l2_spread' not in df.columns:
            return {
                "has_l2_orderbook": False,
                "micro_deviation_sig": False,
                "l2_spread_stationary": False,
                "micro_dev_predictive_power": 0.0
            }

        micro_dev = df['micro_mid_deviation'].fillna(0)
        l2_spread = df['l2_spread'].fillna(0)

        # Tương quan hạng Spearman: Độ lệch Micro-price có dẫn dắt biến động giá không?
        dev_corr, dev_pval = spearmanr(micro_dev.shift(1).fillna(0), r)
        is_dev_sig = bool(dev_pval < 0.05 and abs(dev_corr) > 0.05)

        # Kiểm định tính dừng của L2 Spread (Tránh rủi ro phi dừng/trôi dạt)
        try:
            spread_adf_stat = adfuller(l2_spread, autolag='AIC')[1]
            is_spread_stat = bool(spread_adf_stat < 0.05)
        except Exception:
            is_spread_stat = False

        return {
            "has_l2_orderbook": True,
            "micro_deviation_sig": is_dev_sig,
            "micro_dev_predictive_power": float(dev_corr),
            "l2_spread_stationary": is_spread_stat
        }

class Layer8NonlinearDependence:
    @staticmethod
    def run(returns: pd.Series) -> dict:
        r = returns.dropna()
        if len(r) < 50:
            return {"is_nonlinear_dependent": False, "mutual_information_lag1": 0.0}

        ar_res = AutoReg(r.values, lags=1).fit()
        resid = ar_res.resid
        resid_std = (resid - np.nanmean(resid)) / (np.nanstd(resid) + 1e-8)

        bds_stat, p_val = bds(resid_std, max_dim=2, epsilon=None)

        x_lag = r.shift(1).dropna()
        y_curr = r.iloc[1:]
        mi = mutual_info_regression(x_lag.values.reshape(-1, 1), y_curr.values, random_state=42)[0]

        return {
            "bds_stat_dim2": float(bds_stat),
            "bds_pvalue_dim2": float(p_val),
            "is_nonlinear_dependent": bool(p_val < 0.05),
            "mutual_information_lag1": float(mi)
        }

class Layer8bComplexityDiagnostics:
    """Chẩn đoán tính bất định đa quy mô (Permutation Entropy)."""
    @staticmethod
    def run(returns: pd.Series) -> dict:
        r = returns.dropna().values
        if len(r) < 30:
            return {"is_high_complexity": False, "permutation_entropy": 1.0}

        m, tau = 3, 1
        n = len(r) - (m - 1) * tau
        patterns = np.array([r[i : i + m * tau : tau] for i in range(n)])
        ranks = np.argsort(patterns, axis=1)
        _, counts = np.unique(ranks, axis=0, return_counts=True)
        probs = counts / counts.sum()

        pe = -np.sum(probs * np.log2(probs + 1e-8)) / np.log2(math.factorial(m))

        return {
            "permutation_entropy": float(pe),
            "is_high_complexity": bool(pe > 0.85)
        }

class Layer9RegimeStability:
    """Xác định xác suất trạng thái tiềm ẩn thông qua chuỗi Markov."""
    @staticmethod
    def run(returns: pd.Series) -> pd.DataFrame:
        r = returns.dropna()
        r_numeric = pd.Series(r.values)
        try:
            model = MarkovRegression(r_numeric, k_regimes=2, trend='c', switching_variance=True)
            res = model.fit(disp=False)
            probs = res.filtered_marginal_probabilities
            probs.index = r.index
            probs.columns = ['prob_regime_low_vol', 'prob_regime_high_vol']
            return probs
        except Exception:
            rolling_std = r.rolling(20).std(ddof=1)
            p_high = (rolling_std > rolling_std.median()).astype(float)
            return pd.DataFrame({
                'prob_regime_low_vol': 1.0 - p_high,
                'prob_regime_high_vol': p_high
            }, index=r.index)

### Scanner

In [ ]:
# =====================================================================
# SCANNER TỔNG HỢP (SINGLE-ASSET DIAGNOSTIC SCANNER)
# =====================================================================
class Stage2SingleAssetDGPScanner:
    """
    Quét chẩn đoán kiểm định chuỗi thời gian cho Đơn tài sản (Single-Asset).
    Tập hợp sức mạnh của Vi phân phân số, Động lực học phi tuyến, Bước nhảy và Vi cấu trúc L2
    thành một Payload ủy quyền để định tuyến Feature Engineering (Stage 3).
    """
    @staticmethod
    def execute(df: pd.DataFrame) -> dict:
        returns = df['log_return'].dropna()
        prices = df['close_adj'].dropna()

        # Thực thi quy trình kiểm định thuần túy qua các bài test thống kê
        mem_stats = Layer4MemoryDependence.multi_scale_memory(prices, returns)
        cycle_stats = Layer4bSpectralCycleDiagnostics.run(prices)
        frac_stats = Layer5FractionalIntegration.find_optimal_d(prices)
        vol_stats = Layer6VolatilityDynamics.estimate(df)
        jump_stats = Layer6bVolatilityJumpDiagnostics.run(df)
        volu_stats = Layer7bVolumeDiagnostics.run(df)
        nonlin_stats = Layer8NonlinearDependence.run(returns)
        complex_stats = Layer8bComplexityDiagnostics.run(returns)

        # Nhận diện đặc tính Sổ lệnh vi mô L2
        micro_stats = Layer7cMicrostructureDiagnostics.run(df)

        # Đóng gói Routing Payload
        routing_payload = {
            "dynamic_lags": mem_stats['dynamic_lags'],
            "optimal_d": frac_stats['optimal_d'],
            "dominant_cycle": cycle_stats['dominant_cycle_len'],
            "flags": {
                # Cờ Động lực học OHLCV cốt lõi
                "has_long_trend": mem_stats['long_term_trending'],
                "has_short_reversion": mem_stats['short_term_mean_reverting'],
                "has_vol_clustering": vol_stats['has_arch_effect'],
                "has_asymmetric_vol": vol_stats['has_asymmetric_vol'],
                "is_volume_significant": volu_stats['is_volume_significant'],
                "is_nonlinear": nonlin_stats['is_nonlinear_dependent'],
                "has_vol_jumps": jump_stats['has_volatility_jumps'],
                "is_high_complexity": complex_stats['is_high_complexity'],

                # Cờ Vi cấu trúc (L2 Order Book)
                "has_l2_orderbook": micro_stats['has_l2_orderbook'],
                "micro_deviation_sig": micro_stats['micro_deviation_sig']
            }
        }

        return routing_payload

## BƯỚC 3: QUYẾT ĐỊNH HỌ ĐẶC TRƯNG VÀ THIẾT KẾ KHÔNG GIAN DỮ LIỆU ỨNG VIÊN
Mục tiêu: Dùng bảng quyết định logic (Evidence-based Matrix) để chỉ sinh các họ feature có căn cứ kinh tế lượng, sau đó tính toán theo quy tắc đa quy mô.

In [ ]:
# =====================================================================
# 1. BASE STRATEGY INTERFACE
# =====================================================================
class BaseFeatureStrategy(ABC):
    @abstractmethod
    def construct(self, df: pd.DataFrame, eps: float = 1e-8) -> pd.DataFrame:
        pass

# =====================================================================
# 2. FEATURE STRATEGY MODULES (KINEMATICS, CHAOS, JUMPS)
# =====================================================================

class BaselineStrategy(BaseFeatureStrategy):
    def __init__(self, dynamic_lags: list[int]):
        self.lags = dynamic_lags

    def construct(self, df: pd.DataFrame, eps: float = 1e-8) -> pd.DataFrame:
        feat = pd.DataFrame(index=df.index)
        h, l, c, o = df['high_adj'], df['low_adj'], df['close_adj'], df['open_adj']
        v = df['volume']

        candle_range = (h - l) + eps
        feat['geo_body_ratio'] = (c - o) / candle_range
        feat['geo_upper_shadow'] = (h - np.maximum(o, c)) / candle_range
        feat['geo_lower_shadow'] = (np.minimum(o, c) - l) / candle_range
        feat['overnight_gap'] = (o / c.shift(1)) - 1.0

        log_v = np.log(v + eps)
        for k in self.lags:
            if k >= 3:
                roll_vol_mean = v.shift(1).rolling(k).mean()
                feat[f'rvol_{k}'] = v / (roll_vol_mean + eps)
                mu_lv = log_v.shift(1).rolling(k).mean()
                std_lv = log_v.shift(1).rolling(k).std(ddof=1)
                feat[f'vol_shock_{k}'] = (log_v - mu_lv) / (std_lv + eps)

        if isinstance(df.index, pd.DatetimeIndex):
            dow = df.index.dayofweek
            month = df.index.month
            feat['sin_dow'] = np.sin(2 * np.pi * dow / 5.0)
            feat['cos_dow'] = np.cos(2 * np.pi * dow / 5.0)
            feat['sin_month'] = np.sin(2 * np.pi * month / 12.0)
            feat['cos_month'] = np.cos(2 * np.pi * month / 12.0)

        return feat

class TrendMomentumStrategy(BaseFeatureStrategy):
    def __init__(self, dynamic_lags: list[int]):
        self.lags = dynamic_lags

    def construct(self, df: pd.DataFrame, eps: float = 1e-8) -> pd.DataFrame:
        feat = pd.DataFrame(index=df.index)
        p = df['close_adj']

        for k in self.lags:
            ret_k = np.log(p / p.shift(k))
            feat[f'ret_{k}'] = ret_k
            feat[f'tsmom_sign_{k}'] = np.sign(ret_k)
            if k >= 3:
                roll_max = p.shift(1).rolling(k).max()
                roll_min = p.shift(1).rolling(k).min()
                feat[f'breakout_ratio_{k}'] = (p - roll_max) / ((roll_max - roll_min) + eps)

        ema_12 = p.ewm(span=12, adjust=False).mean()
        ema_26 = p.ewm(span=26, adjust=False).mean()
        feat['macd_dist'] = (ema_12 / ema_26) - 1.0
        feat['macd_slope_5'] = feat['macd_dist'] - feat['macd_dist'].shift(5)
        return feat

class OscillatorMeanReversionStrategy(BaseFeatureStrategy):
    def __init__(self, dynamic_lags: list[int]):
        self.lags = [k for k in dynamic_lags if k <= 20] or [5, 10, 20]

    def construct(self, df: pd.DataFrame, eps: float = 1e-8) -> pd.DataFrame:
        feat = pd.DataFrame(index=df.index)
        p = df['close_adj']

        for k in self.lags:
            if k >= 3:
                roll_mean = p.shift(1).rolling(k).mean()
                roll_std = p.shift(1).rolling(k).std(ddof=1)
                feat[f'zscore_{k}'] = (p - roll_mean) / (roll_std + eps)
                upper_band = roll_mean + (2.0 * roll_std)
                lower_band = roll_mean - (2.0 * roll_std)
                feat[f'bollinger_pctB_{k}'] = (p - lower_band) / ((upper_band - lower_band) + eps)

        delta = p.diff()
        gain = delta.where(delta > 0, 0.0).shift(1).rolling(14).mean()
        loss = -delta.where(delta < 0, 0.0).shift(1).rolling(14).mean()
        rs = gain / (loss + eps)
        feat['rsi_14'] = 100.0 - (100.0 / (1.0 + rs))
        return feat

class FractionalMemoryStrategy(BaseFeatureStrategy):
    def __init__(self, optimal_d: float, threshold: float = 1e-4, max_window: int = 100):
        self.d = optimal_d
        self.threshold = threshold
        self.max_window = max_window
        self.weights_forward = self._compute_weights(max_window)
        self.actual_max_len = len(self.weights_forward)

    def _compute_weights(self, size: int) -> np.ndarray:
        w = [1.0]
        for k in range(1, size):
            w_k = -w[-1] / k * (self.d - k + 1)
            if abs(w_k) < self.threshold:
                break
            w.append(w_k)
        return np.array(w)

    def construct(self, df: pd.DataFrame, eps: float = 1e-8) -> pd.DataFrame:
        feat = pd.DataFrame(index=df.index)
        p = df['close_adj']
        def apply_frac_diff(x):
            x_rev = x[::-1]
            valid_len = min(len(x_rev), self.actual_max_len)
            return np.dot(self.weights_forward[:valid_len], x_rev[:valid_len])

        feat[f'frac_diff_d{self.d:.2f}'] = p.rolling(self.max_window, min_periods=10).apply(apply_frac_diff, raw=True)
        return feat

class VolatilityDynamicsStrategy(BaseFeatureStrategy):
    def __init__(self, dynamic_lags: list[int], has_asymmetric_vol: bool):
        self.lags = dynamic_lags
        self.has_asymmetric = has_asymmetric_vol

    def construct(self, df: pd.DataFrame, eps: float = 1e-8) -> pd.DataFrame:
        feat = pd.DataFrame(index=df.index)
        h, l, c, o = df['high_adj'], df['low_adj'], df['close_adj'], df['open_adj']
        r = df['log_return']

        feat['vol_parkinson'] = np.sqrt((1.0 / (4.0 * np.log(2.0))) * (np.log(h / l) ** 2))
        log_hl = np.log(h / l)
        log_co = np.log(c / o)
        gk_core = 0.5 * (log_hl ** 2) - (2.0 * np.log(2.0) - 1.0) * (log_co ** 2)
        feat['vol_garman_klass'] = np.sqrt(np.maximum(0.0, gk_core))
        rs_core = np.log(h / c) * np.log(h / o) + np.log(l / c) * np.log(l / o)
        feat['vol_rogers_satchell'] = np.sqrt(np.maximum(0.0, rs_core))

        window = 20
        o_prev_c = np.log(o / c.shift(1))
        c_prev_o = np.log(c / o)
        var_o = o_prev_c.rolling(window).var()
        var_c = c_prev_o.rolling(window).var()
        var_rs = pd.Series(rs_core, index=df.index).rolling(window).mean()
        k_val = 0.34 / (1.34 + (window + 1.0) / (window - 1.0))
        feat['vol_yang_zhang'] = np.sqrt(np.maximum(0.0, var_o + k_val * var_c + (1.0 - k_val) * var_rs))

        if self.has_asymmetric:
            downside_ret = r.where(r < 0, 0.0)
            feat['vol_downside_dev_20'] = downside_ret.rolling(20).std(ddof=1)
            feat['vol_upside_dev_20'] = r.where(r > 0, 0.0).rolling(20).std(ddof=1)
            feat['vol_semi_variance_ratio_raw'] = feat['vol_downside_dev_20'] / (feat['vol_upside_dev_20'] + eps)

        feat['skewness_20'] = r.rolling(20).skew()
        feat['kurtosis_20'] = r.rolling(20).kurt()

        return feat

class VolatilityJumpDiffusionStrategy(BaseFeatureStrategy):
    def __init__(self, window: int = 20):
        self.window = window

    def construct(self, df: pd.DataFrame, eps: float = 1e-8) -> pd.DataFrame:
        feat = pd.DataFrame(index=df.index)
        r = df['log_return'].fillna(0)

        rv = (r ** 2).rolling(self.window).sum()
        abs_r = r.abs()
        bv = (np.pi / 2.0) * (abs_r * abs_r.shift(1)).rolling(self.window).sum()

        feat['jump_diffusion_component'] = np.maximum(rv - bv, 0.0) / (rv + eps)
        feat['continuous_vol_ratio'] = bv / (rv + eps)
        feat['jump_signed_shock'] = feat['jump_diffusion_component'] * np.sign(r)
        return feat

class MultiScaleComplexityStrategy(BaseFeatureStrategy):
    def construct(self, df: pd.DataFrame, eps: float = 1e-8) -> pd.DataFrame:
        feat = pd.DataFrame(index=df.index)
        r = df['log_return'].fillna(0)

        def calc_pe(x):
            if len(x) < 6: return np.nan
            m, tau = 3, 1
            n = len(x) - (m - 1) * tau
            patterns = np.array([x[i : i + m * tau : tau] for i in range(n)])
            ranks = np.argsort(patterns, axis=1)
            _, counts = np.unique(ranks, axis=0, return_counts=True)
            probs = counts / counts.sum()
            return -np.sum(probs * np.log2(probs + 1e-8)) / np.log2(6.0)

        feat['chaos_permutation_entropy_20'] = r.rolling(20).apply(calc_pe, raw=True)
        binary_seq = (r > 0).astype(int)
        feat['lz_complexity_proxy_20'] = (binary_seq.diff().abs()).rolling(20).mean()
        return feat

class WaveletMultiResolutionStrategy(BaseFeatureStrategy):
    def construct(self, df: pd.DataFrame, eps: float = 1e-8) -> pd.DataFrame:
        feat = pd.DataFrame(index=df.index)
        p = df['close_adj']

        p_s1 = p.shift(1)
        feat['wavelet_detail_d1'] = (p - p_s1) / np.sqrt(2.0)
        feat['wavelet_detail_d2'] = ((p + p_s1) - (p.shift(2) + p.shift(3))) / 2.0
        feat['wavelet_approx_a3'] = p.rolling(8).mean()
        feat['wavelet_energy_ratio'] = (feat['wavelet_detail_d1'] ** 2) / ((feat['wavelet_detail_d2'] ** 2) + eps)
        return feat

class IntradayShadowPressureStrategy(BaseFeatureStrategy):
    def construct(self, df: pd.DataFrame, eps: float = 1e-8) -> pd.DataFrame:
        feat = pd.DataFrame(index=df.index)
        h, l, c, o = df['high_adj'], df['low_adj'], df['close_adj'], df['open_adj']
        v = df['volume']

        candle_range = (h - l) + eps
        body = (c - o).abs()
        upper_shadow = h - np.maximum(o, c)
        lower_shadow = np.minimum(o, c) - l

        feat['shadow_asymmetry_ratio'] = (upper_shadow - lower_shadow) / candle_range
        feat['buying_tail_power'] = (lower_shadow / candle_range) * v
        feat['selling_tail_power'] = (upper_shadow / candle_range) * v
        feat['body_efficiency_ratio'] = body / candle_range
        return feat

class KinematicDynamicsStrategy(BaseFeatureStrategy):
    def construct(self, df: pd.DataFrame, eps: float = 1e-8) -> pd.DataFrame:
        feat = pd.DataFrame(index=df.index)
        p = df['close_adj']
        h, l = df['high_adj'], df['low_adj']
        v = df['volume']
        r = df['log_return'].fillna(0)

        feat['kinematic_velocity'] = r
        feat['kinematic_acceleration'] = r - r.shift(1)
        feat['kinematic_jerk'] = feat['kinematic_acceleration'] - feat['kinematic_acceleration'].shift(1)

        roll_std = p.shift(1).rolling(20).std(ddof=1)
        bb_width = 4.0 * roll_std

        p_prev = p.shift(1)
        tr1 = h - l
        tr2 = (h - p_prev).abs()
        tr3 = (l - p_prev).abs()
        tr = pd.concat([tr1, tr2, tr3], axis=1).max(axis=1)
        atr = tr.shift(1).rolling(20).mean()
        kc_width = 3.0 * atr

        feat['kinematic_squeeze_ratio'] = bb_width / (kc_width + eps)

        clv = ((p - l) - (h - p)) / (h - l + eps)
        feat['kinematic_clv_vol'] = clv * v
        feat['kinematic_clv_vol_roll20'] = feat['kinematic_clv_vol'].rolling(20).mean()
        return feat

class GMMRegimeStrategy(BaseFeatureStrategy):
    def __init__(self, window: int = 120, update_freq: int = 20):
        self.window = window
        self.update_freq = update_freq

    def construct(self, df: pd.DataFrame, eps: float = 1e-8) -> pd.DataFrame:
        feat = pd.DataFrame(index=df.index)
        r = df['log_return'].fillna(0)
        vol = r.rolling(20).std(ddof=1).fillna(0)

        n = len(df)
        gmm_bull = np.zeros(n)
        gmm_bear = np.zeros(n)
        X = np.column_stack((r.values, vol.values))

        for i in range(self.window, n, self.update_freq):
            start_idx = max(0, i - self.window)
            X_train = np.nan_to_num(X[start_idx:i])
            pred_end = min(n, i + self.update_freq)
            X_test = np.nan_to_num(X[i:pred_end])

            try:
                if np.var(X_train[:, 0]) > 1e-8:
                    gmm = GaussianMixture(n_components=2, random_state=42, n_init=1)
                    gmm.fit(X_train)
                    probs = gmm.predict_proba(X_test)
                    means = gmm.means_
                    bull_idx = np.argmax(means[:, 0])
                    bear_idx = 1 - bull_idx
                    gmm_bull[i:pred_end] = probs[:, bull_idx]
                    gmm_bear[i:pred_end] = probs[:, bear_idx]
            except Exception:
                pass

        feat['gmm_prob_bull'] = gmm_bull
        feat['gmm_prob_bear'] = gmm_bear
        feat.iloc[:self.window, :] = np.nan
        return feat

# =====================================================================
# 3. L2 ORDER BOOK & MICROSTRUCTURE STRATEGIES (ĐÃ NÂNG CẤP)
# =====================================================================

class L2MicrostructureStrategy(BaseFeatureStrategy):
    """Khai thác động lực học vi mô thực tế từ dữ liệu Sổ lệnh (L2 Order Book)."""
    def construct(self, df: pd.DataFrame, eps: float = 1e-8) -> pd.DataFrame:
        feat = pd.DataFrame(index=df.index)

        # 1. Micro-Price Pressure (Áp lực điều hướng giá)
        if 'micro_mid_deviation' in df.columns:
            feat['l2_micro_dev_raw'] = df['micro_mid_deviation']
            feat['l2_micro_dev_accel'] = feat['l2_micro_dev_raw'] - feat['l2_micro_dev_raw'].shift(1)

            # 2. Spread Dynamics (Tính co giãn thanh khoản sổ lệnh)
            if 'l2_spread' in df.columns:
                spread = df['l2_spread']
                feat['l2_spread_zscore'] = (spread - spread.rolling(20).mean()) / (spread.rolling(20).std() + eps)

            # 3. LOB Imbalance, Depth & Replenishment/Cancellation Proxies
            # Tương thích với L2 Đa tầng từ Stage 1 mới (dùng bid_size_1/ask_size_1) hoặc cơ bản
            b_size = df.get('bid_size_1', df.get('bid_size', pd.Series(0, index=df.index)))
            a_size = df.get('ask_size_1', df.get('ask_size', pd.Series(0, index=df.index)))

            if not (b_size.sum() == 0 and a_size.sum() == 0):
                # 3.1 Static LOB Imbalance
                imbalance = b_size - a_size
                feat['l2_order_imbalance_raw'] = imbalance
                feat['l2_order_imbalance_z'] = (imbalance - imbalance.rolling(20).mean()) / (imbalance.rolling(20).std() + eps)

                # 3.2 L2 Depth (Độ sâu thanh khoản Top-of-Book)
                feat['l2_depth_total'] = b_size + a_size
                feat['l2_depth_ratio'] = b_size / (a_size + eps)

                # 3.3 Replenishment & Cancellation Proxy (Động lực học chèn/hủy lệnh)
                delta_b = b_size.diff().fillna(0)
                delta_a = a_size.diff().fillna(0)

                # Tốc độ bổ sung lệnh (Replenishment Rate)
                feat['l2_bid_replenishment'] = np.where(delta_b > 0, delta_b, 0)
                feat['l2_ask_replenishment'] = np.where(delta_a > 0, delta_a, 0)

                # Áp lực rút/hủy lệnh (Withdrawal / Cancellation Proxy)
                feat['l2_bid_withdrawal'] = np.where(delta_b < 0, -delta_b, 0)
                feat['l2_ask_withdrawal'] = np.where(delta_a < 0, -delta_a, 0)

        return feat

class OrderFlowToxicityStrategy(BaseFeatureStrategy):
    """
    Tính toán OFI và VPIN.
    Nếu không có L2 Book, sẽ rơi về chế độ ước lượng (Fallback) thông qua Volume.
    """
    def construct(self, df: pd.DataFrame, eps: float = 1e-8) -> pd.DataFrame:
        feat = pd.DataFrame(index=df.index)

        # Trạng thái 1: Có dữ liệu L2 Order Book thực (Ưu tiên)
        if all(col in df.columns for col in ['bid_size_1', 'ask_size_1']):
            order_imbalance = df['bid_size_1'] - df['ask_size_1']
        elif all(col in df.columns for col in ['bid_size', 'ask_size']):
            order_imbalance = df['bid_size'] - df['ask_size']
        # Trạng thái 2: Fallback ước lượng từ giá đóng cửa/mở cửa (OHLCV)
        else:
            c, o, h, l = df['close_adj'], df['open_adj'], df['high_adj'], df['low_adj']
            v = df['volume']
            buy_pressure = (c - l) / (h - l + eps)
            sell_pressure = (h - c) / (h - l + eps)
            order_imbalance = (buy_pressure - sell_pressure) * v

        feat['flow_imbalance_proxy'] = order_imbalance
        feat['flow_imbalance_zscore'] = (order_imbalance - order_imbalance.rolling(20).mean()) / (order_imbalance.rolling(20).std() + eps)

        v_series = df['volume'] if 'volume' in df.columns else pd.Series(1, index=df.index)
        vol_bucket = v_series.rolling(10).sum()
        imbalance_bucket = order_imbalance.abs().rolling(10).sum()
        feat['flow_vpin_10'] = imbalance_bucket / (vol_bucket + eps)

        return feat

class LiquidityMicrostructureStrategy(BaseFeatureStrategy):
    def construct(self, df: pd.DataFrame, eps: float = 1e-8) -> pd.DataFrame:
        feat = pd.DataFrame(index=df.index)
        h, l, c = df['high'], df['low'], df['close']
        v = df['volume']
        r = df['log_return']

        h_prev = h.shift(1)
        l_prev = l.shift(1)
        gamma = (np.log(h / l) ** 2) + (np.log(h_prev / l_prev) ** 2)
        h_2d = np.maximum(h, h_prev)
        l_2d = np.minimum(l, l_prev)
        beta = np.log(h_2d / l_2d) ** 2
        den = 3.0 - 2.0 * np.sqrt(2.0)
        alpha = (np.sqrt(2.0 * beta) - np.sqrt(beta)) / den - np.sqrt(gamma / den)
        spread = 2.0 * (np.exp(alpha) - 1.0) / (1.0 + np.exp(alpha))
        feat['liq_spread_cs'] = spread.clip(lower=0.0)

        dollar_volume = c * v
        amihud_raw = r.abs() / dollar_volume.replace(0, np.nan)
        feat['liq_amihud_raw'] = amihud_raw
        feat['liq_amihud_z'] = (amihud_raw - amihud_raw.rolling(20).mean()) / (amihud_raw.rolling(20).std() + eps)

        def calc_autocorr_lag1(x):
            return pd.Series(x).autocorr(lag=1) if len(x) > 2 else 0.0

        feat['liq_roll_measure_20'] = r.rolling(20).apply(calc_autocorr_lag1, raw=False)
        feat['liq_log_turnover'] = np.log(dollar_volume + eps)
        feat['liq_turnover_ratio_20'] = dollar_volume / (dollar_volume.shift(1).rolling(20).mean() + eps)
        return feat

# =====================================================================
# 4. FEATURE STRATEGY MODULES (SINGLE-ASSET + MICROSTRUCTURE)
# =====================================================================

class NonlinearInteractionStrategy(BaseFeatureStrategy):
    def construct(self, df: pd.DataFrame, eps: float = 1e-8) -> pd.DataFrame:
        feat = pd.DataFrame(index=df.index)
        r = df['log_return']
        h, l = df['high_adj'], df['low_adj']

        feat['nonlin_ret_squared_signed'] = (r ** 2) * np.sign(r)
        feat['nonlin_ret_cubed'] = r ** 3
        daily_range = np.log(h / l)
        feat['nonlin_range_x_ret'] = daily_range * r

        vol_roll_mean = df['volume'].shift(1).rolling(20).mean()
        feat['nonlin_ret_x_volshock'] = r * (df['volume'] / (vol_roll_mean + eps))
        return feat

class RegimeConditionedStrategy(BaseFeatureStrategy):
    def construct(self, df: pd.DataFrame, eps: float = 1e-8) -> pd.DataFrame:
        feat = pd.DataFrame(index=df.index)
        r = df['log_return']

        roll_vol = r.rolling(20).std(ddof=1)
        vol_q50 = roll_vol.shift(1).rolling(60).median()
        p_high_vol = (roll_vol > vol_q50).astype(float)

        feat['regime_p_high_vol'] = p_high_vol
        feat['regime_weighted_return'] = r * p_high_vol
        feat['regime_transition_shock'] = p_high_vol.diff().abs()
        return feat

class MarketEfficiencyStrategy(BaseFeatureStrategy):
    def construct(self, df: pd.DataFrame, eps: float = 1e-8) -> pd.DataFrame:
        feat = pd.DataFrame(index=df.index)
        p = df['close_adj']
        r = df['log_return'].fillna(0)

        vol_5 = r.rolling(5).std(ddof=1)
        vol_20 = r.rolling(20).std(ddof=1)
        feat['chaos_hurst_proxy'] = np.log(vol_20 / (vol_5 + eps)) / np.log(4.0)

        def calc_entropy(x):
            if len(x) < 10: return np.nan
            hist, _ = np.histogram(x, bins=10, density=True)
            p_data = hist[hist > 0]
            return entropy(p_data)

        feat['chaos_shannon_entropy_20'] = r.rolling(20).apply(calc_entropy, raw=True)
        net_change = p.diff(10).abs()
        sum_abs_change = p.diff().abs().rolling(10).sum()
        feat['chaos_kaufman_er_10'] = net_change / (sum_abs_change + eps)
        return feat

# =====================================================================
# 5. ADVANCED INSTITUTIONAL MODULES
# =====================================================================

class AsymmetricStressStrategy(BaseFeatureStrategy):
    def __init__(self, window: int = 20):
        self.window = window

    def construct(self, df: pd.DataFrame, eps: float = 1e-8) -> pd.DataFrame:
        feat = pd.DataFrame(index=df.index)
        r = df['log_return'].fillna(0)

        r_pos = np.maximum(r, 0)
        r_neg = np.minimum(r, 0)
        vol_up = pd.Series(r_pos, index=df.index).rolling(self.window).std(ddof=1)
        vol_down = pd.Series(r_neg, index=df.index).rolling(self.window).std(ddof=1)
        feat['asym_ratio_20'] = vol_up / (vol_down + eps)

        synth_price = (1 + r).cumprod()
        rolling_max = synth_price.rolling(self.window).max()
        feat['drawdown_20'] = (synth_price - rolling_max) / (rolling_max + eps)

        feat['drawdown_velocity'] = feat['drawdown_20'] - feat['drawdown_20'].shift(1)
        return feat

class VolatilityTermStructureStrategy(BaseFeatureStrategy):
    def construct(self, df: pd.DataFrame, eps: float = 1e-8) -> pd.DataFrame:
        feat = pd.DataFrame(index=df.index)
        r = df['log_return'].fillna(0)

        vol_short = r.rolling(5).std(ddof=1)
        vol_long = r.rolling(20).std(ddof=1)
        feat['vts_5_20'] = vol_short / (vol_long + eps)

        feat['vol_accel'] = feat['vts_5_20'] - feat['vts_5_20'].shift(1)
        return feat

class VWAPPressureStrategy(BaseFeatureStrategy):
    def __init__(self, window: int = 20):
        self.window = window

    def construct(self, df: pd.DataFrame, eps: float = 1e-8) -> pd.DataFrame:
        feat = pd.DataFrame(index=df.index)
        h, l, c = df['high_adj'], df['low_adj'], df['close_adj']
        v = df['volume']

        tp = (h + l + c) / 3.0
        tp_v = tp * v
        roll_tp_v = tp_v.rolling(self.window).sum()
        roll_v = v.rolling(self.window).sum()
        vwap = roll_tp_v / (roll_v + eps)

        feat['vwap_distance_20'] = (c - vwap) / (vwap + eps)
        return feat

class VolumePriceDivergenceStrategy(BaseFeatureStrategy):
    """
    Đo lường phân kỳ động lượng giữa giá và dòng tiền (Flow Divergence).
    [ĐÃ SỬA LỖI LEAKAGE]: Thay thế toàn bộ cumsum() toàn cục bằng Rolling Window tĩnh.
    """
    def __init__(self, window: int = 20):
        self.window = window

    def construct(self, df: pd.DataFrame, eps: float = 1e-8) -> pd.DataFrame:
        feat = pd.DataFrame(index=df.index)
        h, l, c, o = df['high_adj'], df['low_adj'], df['close_adj'], df['open_adj']
        v = df['volume']
        r = df['log_return'].fillna(0)

        # 1. Chaikin Money Flow (CMF) Proxy (Chỉ dùng Rolling, Không rò rỉ)
        clv = ((c - l) - (h - c)) / ((h - l) + eps)
        vol_money_flow = clv * v
        feat['cmf_20'] = vol_money_flow.rolling(self.window).sum() / (v.rolling(self.window).sum() + eps)
        feat['cmf_slope_5'] = feat['cmf_20'] - feat['cmf_20'].shift(5)

        # 2. Rolling On-Balance Volume (OBV) Momentum (Thay cumsum = rolling sum)
        obv_flow = np.sign(r) * v
        roll_obv = obv_flow.rolling(self.window).sum()
        roll_obv_std = obv_flow.rolling(self.window).std(ddof=1) + eps
        feat['obv_zscore_20'] = roll_obv / roll_obv_std
        feat['obv_slope_5'] = (roll_obv - obv_flow.shift(5).rolling(self.window).sum()) / (v.rolling(self.window).mean() + eps)

        # 3. Rolling Volume-Price Trend (VPT) Divergence (Thay cumsum = rolling sum)
        vpt_flow = r * v
        roll_vpt = vpt_flow.rolling(self.window).sum()
        roll_vpt_std = vpt_flow.rolling(self.window).std(ddof=1) + eps
        feat['vpt_zscore_20'] = roll_vpt / roll_vpt_std

        # 4. Phân kỳ trực tiếp giữa Price Return và CMF
        norm_price_ret = (c / c.shift(self.window) - 1.0)
        feat['cmf_price_divergence'] = feat['cmf_20'] - norm_price_ret

        return feat


class DirectionalPressureStrategy(BaseFeatureStrategy):
    """
    Đo lường áp lực định hướng thông qua cấu trúc nến có trọng số thanh khoản (Price Action Pressure).
    """
    def __init__(self, window: int = 14):
        self.window = window

    def construct(self, df: pd.DataFrame, eps: float = 1e-8) -> pd.DataFrame:
        feat = pd.DataFrame(index=df.index)
        h, l, c, o = df['high_adj'], df['low_adj'], df['close_adj'], df['open_adj']
        v = df['volume']

        candle_range = (h - l) + eps
        upper_wick = h - np.maximum(o, c)
        lower_wick = np.minimum(o, c) - l
        body = c - o

        # 1. Buying vs Selling Pressure Ratio
        # Áp lực mua: Lực đẩy từ đáy + Thân tăng
        buy_strength = (c - l) / candle_range
        sell_strength = (h - c) / candle_range
        feat['net_pressure_ratio'] = buy_strength - sell_strength

        # 2. Volume-Weighted Pressure Intensity
        feat['net_pressure_volume'] = feat['net_pressure_ratio'] * np.log(v + 1.0)
        feat['net_pressure_vol_roll'] = feat['net_pressure_volume'].rolling(self.window).mean()

        # 3. Asymmetric Wick Absorption
        # Đuôi nến dưới áp đảo đuôi nến trên thể hiện sự hấp thụ cung (Bullish rejection)
        feat['wick_absorption_direction'] = (lower_wick - upper_wick) / candle_range
        feat['wick_absorption_thrust'] = feat['wick_absorption_direction'] * (v / (v.rolling(self.window).mean() + eps))

        # 4. Body Acceleration Vector (Định hướng bứt phá)
        body_ratio = body / candle_range
        feat['body_direction_momentum'] = body_ratio.rolling(5).mean()
        feat['body_direction_accel'] = feat['body_direction_momentum'] - feat['body_direction_momentum'].shift(2)

        return feat


class SignedVolatilityDivergenceStrategy(BaseFeatureStrategy):
    """
    Tách biệt rủi ro giảm (Downside Vol) và rủi ro tăng (Upside Vol) để lượng hóa tính bất đối xứng.
    """
    def __init__(self, window: int = 20):
        self.window = window

    def construct(self, df: pd.DataFrame, eps: float = 1e-8) -> pd.DataFrame:
        feat = pd.DataFrame(index=df.index)
        r = df['log_return'].fillna(0)

        # 1. Tách biệt chuỗi Return âm/dương
        downside_ret = r.where(r < 0, 0.0)
        upside_ret = r.where(r > 0, 0.0)

        # 2. Semi-Variances
        downside_var = (downside_ret ** 2).rolling(self.window).mean()
        upside_var = (upside_ret ** 2).rolling(self.window).mean()

        # 3. Volatility Directional Skew (Độ lệch biến động định hướng)
        # > 0 khi lực tăng chiếm ưu thế, < 0 khi lực giảm chiếm ưu thế
        feat['vol_directional_bias'] = (upside_var - downside_var) / (upside_var + downside_var + eps)

        # 4. Acceleration của Directional Volatility Bias
        feat['vol_directional_accel'] = feat['vol_directional_bias'] - feat['vol_directional_bias'].shift(3)

        # 5. Dynamic High-Low Range Polarization
        c, o = df['close_adj'], df['open_adj']
        signed_body = np.sign(c - o)
        daily_range = np.log(df['high_adj'] / df['low_adj'])
        feat['signed_expansion_thrust'] = signed_body * daily_range

        return feat

class MarketContextStrategy(BaseFeatureStrategy):
    """
    Khai thác tín hiệu ngoại sinh từ chỉ số thị trường chung (VN-Index).
    [ĐÃ SỬA LỖI LEAKAGE]: Ép buộc Shift(1) cho toàn bộ dữ liệu thị trường.
    """
    def __init__(self, window_short: int = 20, window_long: int = 60):
        self.ws = window_short
        self.wl = window_long

    def construct(self, df: pd.DataFrame, eps: float = 1e-8) -> pd.DataFrame:
        feat = pd.DataFrame(index=df.index)

        if 'mkt_return' not in df.columns:
            return feat

        r_asset = df['log_return'].fillna(0)
        # BẮT BUỘC LAG THỊ TRƯỜNG: Đảm bảo thông tin thị trường đã kết thúc
        r_mkt_lag = df['mkt_return'].shift(1).fillna(0)

        # 1. Relative Strength (Alpha)
        feat['mkt_relative_return'] = r_asset - r_mkt_lag
        feat['mkt_rs_cum_20'] = (r_asset.rolling(self.ws).sum() - r_mkt_lag.rolling(self.ws).sum())
        feat['mkt_rs_momentum'] = feat['mkt_rs_cum_20'] - feat['mkt_rs_cum_20'].shift(5)

        # 2. Rolling Beta động
        cov_60 = r_asset.rolling(self.wl).cov(r_mkt_lag)
        var_mkt_60 = r_mkt_lag.rolling(self.wl).var()
        feat['mkt_rolling_beta_60'] = (cov_60 / (var_mkt_60 + eps)).clip(lower=-1.0, upper=3.5)
        feat['mkt_beta_shock'] = feat['mkt_rolling_beta_60'] - feat['mkt_rolling_beta_60'].shift(5)

        # 3. Các chỉ báo tương quan
        vol_asset_20 = r_asset.rolling(self.ws).std(ddof=1)
        vol_mkt_20 = r_mkt_lag.rolling(self.ws).std(ddof=1)
        feat['mkt_vol_ratio_20'] = vol_asset_20 / (vol_mkt_20 + eps)
        feat['mkt_correlation_20'] = r_asset.rolling(self.ws).corr(r_mkt_lag).fillna(0)
        feat['mkt_divergence_signed'] = np.sign(r_asset) * np.sign(r_mkt_lag) * (r_asset - r_mkt_lag)

        return feat

### Router Functions:

In [ ]:
# =====================================================================
# 6. ROUTER  Layer 10 (dou-layers) TỰ ĐỘNG (SINGLE-ASSET + L2 MICROSTRUCTURE)
# =====================================================================
class StatelessFeatureRouter:
    """
    Chạy đúng 1 lần trên toàn bộ chuỗi lịch sử trước khi vào CV.
    [ĐÃ SỬA LỖI LEAKAGE]:
      1. Loại bỏ GMMRegimeStrategy (Chuyển vào Stateful CV).
      2. Tích hợp Shift-Lock toàn cục để triệt tiêu lookahead bias.
    """
    def __init__(self, default_lags=[1, 3, 5, 10, 20], enforce_shift: bool = True):
        self.enforce_shift = enforce_shift
        self.strategies = {
            "Baseline": BaselineStrategy(dynamic_lags=default_lags),
            "Trend": TrendMomentumStrategy(dynamic_lags=default_lags),
            "Oscillator": OscillatorMeanReversionStrategy(dynamic_lags=default_lags),
            "Volatility": VolatilityDynamicsStrategy(dynamic_lags=default_lags, has_asymmetric_vol=True),
            "Kinematics": KinematicDynamicsStrategy(),
            "Wavelet": WaveletMultiResolutionStrategy(),
            "Intraday": IntradayShadowPressureStrategy(),
            "FlowDivergence": VolumePriceDivergenceStrategy(window=20),
            "Directional": DirectionalPressureStrategy(window=14),
            "SignedVol": SignedVolatilityDivergenceStrategy(window=20),
            "Liquidity": LiquidityMicrostructureStrategy(),
            "MarketContext": MarketContextStrategy(window_short=20, window_long=60),
            # GMMRegimeStrategy ĐÃ BỊ XÓA Ở ĐÂY
        }

    def execute(self, df: pd.DataFrame) -> pd.DataFrame:
        feature_frames = []
        for name, strategy in self.strategies.items():
            feat = strategy.construct(df)
            feature_frames.append(feat)

        X_features = pd.concat(feature_frames, axis=1)

        # CHỐT CHẶN AN TOÀN (POINT-IN-TIME LOCK)
        # Toàn bộ đặc trưng vừa sinh ra sẽ lùi lại 1 phiên.
        # Nghĩa là X_features tại dòng 't' chỉ chứa dữ liệu đến hết 't-1'.
        if self.enforce_shift:
            X_features = X_features.shift(1)

        return pd.concat([df, X_features], axis=1)

class StatefulFeaturePipeline(BaseEstimator, TransformerMixin):
    """Chỉ fit tham số GMM, FracDiff d* và thực hiện Feature Selection trên tập Train."""
    def __init__(self, target_horizon=5):
        self.target_horizon = target_horizon
        self.gmm_model = None
        self.bull_idx = None
        self.optimal_d = 1.0
        self.s4_engine = None

    def fit(self, X_train: pd.DataFrame, y_train: pd.Series):
        # 1. Học d* tối ưu trên Train
        frac_res = Layer5FractionalIntegration.find_optimal_d(X_train['close_adj'])
        self.optimal_d = frac_res['optimal_d']

        # 2. Học GMM Regime trên Train
        r = X_train['log_return'].fillna(0).values
        vol = X_train['vol_parkinson'].fillna(0).values # hoặc rolling std
        train_features = np.column_stack([r, vol])

        self.gmm_model = GaussianMixture(n_components=2, random_state=42, n_init=1)
        self.gmm_model.fit(train_features)
        self.bull_idx = np.argmax(self.gmm_model.means_[:, 0])

        # 3. Sinh các cột stateful cho Train
        X_train_stateful = self._compute_stateful_features(X_train)
        X_full_train = pd.concat([X_train, X_train_stateful], axis=1)

        # 4. Fit Stage 4 Selection (VIF, HRP, Granger/MI)
        self.s4_engine = Stage4SelectionEngine(payload={'optimal_d': self.optimal_d})
        self.s4_engine.fit(X_full_train, y_train)
        return self

    def _compute_stateful_features(self, df: pd.DataFrame) -> pd.DataFrame:
        feat = pd.DataFrame(index=df.index)

        # Áp dụng d* đã đóng băng
        if 0.0 < self.optimal_d < 1.0:
            frac_strat = FractionalMemoryStrategy(optimal_d=self.optimal_d)
            feat = pd.concat([feat, frac_strat.construct(df)], axis=1)

        # Áp dụng GMM đã fit (không gọi .fit lại)
        r = df['log_return'].fillna(0).values
        vol = df['vol_parkinson'].fillna(0).values
        probs = self.gmm_model.predict_proba(np.column_stack([r, vol]))
        feat['gmm_prob_bull'] = probs[:, self.bull_idx]
        feat['gmm_prob_bear'] = probs[:, 1 - self.bull_idx]
        return feat

    def transform(self, X: pd.DataFrame) -> pd.DataFrame:
        X_stateful = self._compute_stateful_features(X)
        X_full = pd.concat([X, X_stateful], axis=1)
        return self.s4_engine.transform(X_full)

## BƯỚC 4: KHỬ DƯ THỪA DỮ LIỆU VÀ ĐÁNH GIÁ SỨC MẠNH DỰ BÁO BIÊN
Mục tiêu: Nạp target $y_{t+h}$ vào để thanh lọc các biến cộng tuyến và đo lường lượng thông tin alpha thực chất.

In [ ]:
# =====================================================================
# LAYER 12: REDUNDANCY CONTROL (KHỬ DƯ THỪA TRÊN SINGLE ASSET)
# Đảm bảo tập đặc trưng trực giao (Orthogonal) bằng thống kê
# =====================================================================
class Layer12RedundancyControl:
    @staticmethod
    def linear_vif_prune(X: pd.DataFrame, threshold: float = 0.85, max_vif: float = 5.0) -> pd.DataFrame:
        """Nhánh Tuyến tính: Khử tương quan cặp và Đa cộng tuyến (VIF).

        - Ép kiểu số float64 an toàn, loại bỏ cột phi số và hằng số.
        - Khử tương quan cặp (Pearson correlation).
        - Khử đa cộng tuyến lặp (VIF) dùng ma trận tương quan tối ưu tốc độ.
        """
        if X.empty or X.shape[1] <= 1:
            return X.copy()

        # 1. Ép kiểu số và dọn dẹp các cột không hợp lệ
        X_num = X.copy()
        for col in X_num.columns:
            X_num[col] = pd.to_numeric(X_num[col], errors='coerce')

        # Loại bỏ các cột toàn NaN sau khi ép kiểu
        X_num = X_num.dropna(axis=1, how='all')
        if X_num.shape[1] <= 1:
            return X_num.astype(np.float64)

        # Xử lý NaN cục bộ nếu còn sót lại (forward/backward fill và điền 0)
        X_num = X_num.bfill().ffill().fillna(0.0).astype(np.float64)

        # Loại bỏ các cột không có phương sai (hằng số) để tránh lỗi chia cho 0
        variances = X_num.var(axis=0, ddof=0)
        valid_var_cols = variances[variances > 1e-8].index.tolist()
        X_curr = X_num[valid_var_cols].copy()

        if X_curr.shape[1] <= 1:
            return X_curr

        # 2. Khử tương quan cặp (Pearson)
        corr_matrix = X_curr.corr().abs()
        upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
        to_drop_corr = [col for col in upper.columns if any(upper[col] > threshold)]
        X_curr = X_curr.drop(columns=to_drop_corr)

        # 3. Khử VIF lặp (Sử dụng đường chéo của ma trận nghịch đảo tương quan để tăng tốc)
        while X_curr.shape[1] > 1:
            cols = list(X_curr.columns)
            corr_mat = np.corrcoef(X_curr.values, rowvar=False)

            # Trường hợp có NaN phát sinh trong correlation
            if np.isnan(corr_mat).any():
                vifs = []
                X_vals = X_curr.values
                for j in range(len(cols)):
                    y_target = X_vals[:, j]
                    X_other = np.delete(X_vals, j, axis=1)
                    X_mat = np.column_stack([np.ones(len(X_other)), X_other])
                    try:
                        beta, _, _, _ = np.linalg.lstsq(X_mat, y_target, rcond=None)
                        preds = X_mat @ beta
                        ss_tot = np.sum((y_target - np.mean(y_target)) ** 2)
                        ss_res = np.sum((y_target - preds) ** 2)
                        r2 = 1.0 - (ss_res / (ss_tot + 1e-9))
                        vif = 1.0 / (1.0 - np.clip(r2, 0, 0.9999))
                    except Exception:
                        vif = max_vif + 1.0
                    vifs.append(vif)
            else:
                # VIF của biến chuẩn hóa = phần tử đường chéo của Inverse Correlation Matrix
                try:
                    inv_corr = np.linalg.pinv(corr_mat)
                    vifs = np.diag(inv_corr).tolist()
                except Exception:
                    # Fallback về lstsq từng biến nếu pinv lỗi
                    vifs = [max_vif + 1.0] * len(cols)

            max_vif_val = max(vifs)
            if max_vif_val > max_vif:
                drop_idx = int(np.argmax(vifs))
                X_curr = X_curr.drop(columns=[cols[drop_idx]])
            else:
                break

        return X_curr

    @staticmethod
    def nonlinear_hrp_prune(X: pd.DataFrame, max_clusters: int = 15) -> pd.DataFrame:
        """Nhánh Phi tuyến: Gom cụm phân cấp sử dụng tương quan hạng Spearman"""
        if X.shape[1] <= max_clusters:
            return X

        # Sử dụng Spearman để đo lường các quan hệ phụ thuộc đơn điệu phi tuyến
        corr = X.corr(method='spearman').clip(-1.0, 1.0).fillna(0)
        dist = np.sqrt(np.clip(0.5 * (1.0 - corr), 0, None))

        dist_sym = (dist + dist.T) / 2.0
        np.fill_diagonal(dist_sym.values, 0)

        condensed_dist = squareform(dist_sym, checks=False)
        link = linkage(condensed_dist, method='ward')
        clusters = fcluster(link, t=max_clusters, criterion='maxclust')

        selected_medoids = []
        for c_id in np.unique(clusters):
            cluster_cols = X.columns[clusters == c_id]
            if len(cluster_cols) == 1:
                selected_medoids.append(cluster_cols[0])
            else:
                sub_dist = dist_sym.loc[cluster_cols, cluster_cols]
                # Chọn feature nằm ở trung tâm cụm (medoid)
                medoid = sub_dist.sum(axis=1).idxmin()
                selected_medoids.append(medoid)

        return X[selected_medoids]

# =====================================================================
# LAYER 13: PREDICTIVE DIAGNOSTICS & CAUSALITY
# Lọc nhân quả bằng Granger (Tuyến tính) và Lagged MI (Phi tuyến)
# =====================================================================
class Layer13PredictiveDiagnostics:
    @staticmethod
    def causality_screen(X: pd.DataFrame, y: pd.Series, max_lag: int = 5, p_threshold: float = 0.05) -> tuple:
        """
        Quét nhân quả kép: Kiểm định Granger (tuyến tính) và Lagged Mutual Information (phi tuyến).
        Tính năng được giữ lại nếu vượt qua ít nhất một trong hai bài test thống kê.
        """
        valid_features = []
        optimal_lags = {}

        df_test = pd.concat([y, X], axis=1).dropna()
        if df_test.empty:
            return valid_features, optimal_lags

        y_col = df_test.columns[0]
        y_vals = df_test[y_col].values

        for col in X.columns:
            test_data = df_test[[y_col, col]]
            best_lag = 1
            is_causal = False

            # 1. Granger Causality Test (Tuyến tính)
            best_p = 1.0
            try:
                gc_res = grangercausalitytests(test_data, maxlag=max_lag, verbose=False)
                for lag in range(1, max_lag + 1):
                    p_val = gc_res[lag][0]['ssr_ftest'][1]
                    if p_val < best_p:
                        best_p = p_val
                        best_lag = lag
                if best_p < p_threshold:
                    is_causal = True
            except Exception:
                pass

            # 2. Transfer Entropy Proxy qua Lagged Mutual Information (Phi tuyến)
            if not is_causal:
                best_mi = 0.0
                for lag in range(1, max_lag + 1):
                    x_lagged = df_test[col].shift(lag).fillna(0).values
                    mi_score = mutual_info_classif(x_lagged.reshape(-1, 1), y_vals, random_state=42)[0]
                    if mi_score > best_mi:
                        best_mi = mi_score
                        best_lag = lag

                # Ngưỡng kỳ vọng: MI > 0.01 thể hiện có sự truyền tải thông tin
                if best_mi > 0.01:
                    is_causal = True

            if is_causal:
                valid_features.append(col)
                optimal_lags[col] = best_lag

        # Fallback an toàn bằng Spearman tương quan hạng (Tránh mất hết đặc trưng)
        if len(valid_features) < 3:
            corr_with_y = df_test.drop(columns=[y_col]).corrwith(df_test[y_col], method='spearman').abs()
            top_fallback = corr_with_y.sort_values(ascending=False).head(5).index.tolist()
            for f in top_fallback:
                if f not in valid_features:
                    valid_features.append(f)
                    optimal_lags[f] = 1

        return valid_features, optimal_lags

    @staticmethod
    def regime_conditioned_mi(X: pd.DataFrame, y: pd.Series, prob_high_vol: pd.Series, mi_thresh: float = 0.01) -> tuple:
        """Sàng lọc sức mạnh thông tin dự báo phân rã theo pha thị trường"""
        mask_high_vol = prob_high_vol > 0.5
        mask_low_vol = prob_high_vol <= 0.5

        mi_high = mutual_info_classif(X[mask_high_vol], y[mask_high_vol], random_state=42)
        mi_low = mutual_info_classif(X[mask_low_vol], y[mask_low_vol], random_state=42)
        mi_all = mutual_info_classif(X, y, random_state=42)

        valid_features = []
        results = []

        for idx, col in enumerate(X.columns):
            is_accepted = (mi_high[idx] > mi_thresh) or (mi_low[idx] > mi_thresh)
            if is_accepted:
                valid_features.append(col)

            results.append({
                'feature': col,
                'mi_high_vol': mi_high[idx],
                'mi_low_vol': mi_low[idx],
                'mi_all': mi_all[idx],
                'accepted': is_accepted
            })

        return pd.DataFrame(results).set_index('feature'), valid_features

# =====================================================================
# LAYER 14: KINEMATIC & VOLATILITY-SCALED LAG TRANSFORM
# =====================================================================
class Layer14LagTransformEngine:
    @staticmethod
    def apply_volatility_scaled_lags(X: pd.DataFrame, optimal_lags: dict, eps: float = 1e-8) -> pd.DataFrame:
        X_transformed = pd.DataFrame(index=X.index)

        for col, lag in optimal_lags.items():
            if col not in X.columns:
                continue
            feature_series = X[col]

            # Dịch chuyển chuỗi dựa trên độ trễ tối ưu thống kê (causal lag)
            feature_lagged = feature_series.shift(lag)
            roll_mean = feature_lagged.rolling(20).mean()
            roll_std = feature_lagged.rolling(20).std(ddof=1)

            z_scaled = (feature_lagged - roll_mean) / (roll_std + eps)
            X_transformed[f'{col}_zscaled_lag{lag}'] = z_scaled

            momentum = (feature_series - feature_lagged) / (feature_lagged.abs() + eps)
            acceleration = momentum - momentum.shift(1)

            X_transformed[f'{col}_momentum'] = momentum
            X_transformed[f'{col}_acceleration'] = acceleration

        return X_transformed

### Router Functions:

In [ ]:
# =====================================================================
# STAGE 4 ROUTER: CHUỖI ĐIỀU PHỐI BỘ LỌC THỐNG KÊ & DIRECTIONAL ANCHORING
# =====================================================================
class Stage4ConsensusSelectionRouter:
    """
    Consensus Feature Selection qua Moving Block Subsampling nội bộ trên tập Train.

    Quy trình:
    1. Trích xuất B cửa sổ khối thời gian liên tục (Moving Blocks) trên Train Fold.
    2. Chạy chuỗi kiểm định thống kê đa tầng (VIF -> HRP -> Causality/MI) trên từng block.
    3. Tính điểm đồng thuận (Consensus Rate) và lọc các đặc trưng đạt ngưỡng (ví dụ >= 60%).
    4. Xác định độ trễ tối ưu (Optimal Lag) bằng giá trị trung vị (Median Lag) qua các block.
    """
    def __init__(self, payload: dict, n_blocks: int = 8, block_ratio: float = 0.75, consensus_thresh: float = 0.60):
        self.payload = payload
        self.max_clusters = payload.get('max_clusters', 14)
        self.mi_threshold = payload.get('mi_threshold', 0.01)
        self.min_features = 6
        self.max_features = 16

        # Tham số Consensus Block Subsampling
        self.n_blocks = n_blocks                      # Số lượng khối con lấy mẫu
        self.block_ratio = block_ratio                # Độ dài mỗi khối = 75% chiều dài Train Fold
        self.consensus_thresh = consensus_thresh      # Ngưỡng đồng thuận (tỷ lệ phiếu bầu)

        self.directional_priority_keywords = [
            'mkt_relative', 'mkt_rs', 'mkt_rolling_beta', 'mkt_divergence',
            'cmf', 'obv', 'vpt', 'net_pressure', 'vol_directional',
            'signed_expansion', 'liq_amihud', 'kurtosis_20'
        ]

    def _run_single_filter_pipeline(self, X_sub: pd.DataFrame, y_sub: pd.Series) -> tuple[list[str], dict]:
        """Quy trình sàng lọc thống kê chạy trên 1 block con."""
        # 1. Khử Tuyến tính VIF & Tương quan
        X_pruned = Layer12RedundancyControl.linear_vif_prune(X_sub, threshold=0.85, max_vif=5.0)

        # 2. Khử Phi tuyến HRP Spearman
        X_hrp = Layer12RedundancyControl.nonlinear_hrp_prune(X_pruned, max_clusters=self.max_clusters)

        if 'regime_p_high_vol' in X_sub.columns and 'regime_p_high_vol' not in X_hrp.columns:
            X_hrp['regime_p_high_vol'] = X_sub['regime_p_high_vol']

        # 3. Sàng lọc Nhân quả (Granger + Lagged MI)
        try:
            valid_feats, lags = Layer13PredictiveDiagnostics.causality_screen(X_hrp, y_sub)
        except Exception:
            valid_feats, lags = list(X_hrp.columns), {}

        valid_in_hrp = [c for c in valid_feats if c in X_hrp.columns]
        if len(valid_in_hrp) < self.min_features:
            valid_in_hrp = list(X_hrp.columns)

        return valid_in_hrp, lags

    def execute(self, X_train: pd.DataFrame, y_train: pd.Series) -> tuple:
        # X_train đã được làm sạch rò rỉ hoàn toàn từ _prepare_target phía trước
        X_mat = X_train.copy()

        # 1. Ép kiểu số và loại bỏ các cột rỗng > 30%
        for c in X_mat.columns:
            X_mat[c] = pd.to_numeric(X_mat[c], errors='coerce')

        valid_sparsity_cols = [c for c in X_mat.columns if X_mat[c].isna().sum() < len(X_mat) * 0.3]
        X_mat = X_mat[valid_sparsity_cols].bfill().ffill()

        common_idx = X_mat.index.intersection(y_train.dropna().index)
        X_mat = X_mat.loc[common_idx]
        y_mat = y_train.loc[common_idx]

        n_samples = len(X_mat)
        block_len = max(int(n_samples * self.block_ratio), 40)

        feature_votes = defaultdict(int)
        feature_lags_collected = defaultdict(list)

        print(f"   -> Bắt đầu Consensus Selection qua {self.n_blocks} Moving Blocks (Kích thước block: {block_len}/{n_samples} phiên)...")

        # 2. Sinh các khối thời gian liên tục (Moving / Sliding Blocks)
        max_start = n_samples - block_len
        step = max(1, max_start // max(1, self.n_blocks - 1)) if max_start > 0 else 1
        block_starts = [min(i * step, max_start) for i in range(self.n_blocks)]
        block_starts = sorted(list(set(block_starts)))

        for b_idx, start in enumerate(block_starts):
            end = start + block_len
            X_b = X_mat.iloc[start:end].copy()
            y_b = y_mat.iloc[start:end].copy()

            if len(np.unique(y_b)) < 2:
                continue

            selected_b, lags_b = self._run_single_filter_pipeline(X_b, y_b)

            for feat in selected_b:
                feature_votes[feat] += 1
                if feat in lags_b:
                    feature_lags_collected[feat].append(lags_b[feat])

        # 3. Tính điểm đồng thuận và sàng lọc
        total_valid_blocks = len(block_starts)
        vote_threshold = int(np.ceil(total_valid_blocks * self.consensus_thresh))

        consensus_features = [
            f for f, count in feature_votes.items() if count >= vote_threshold
        ]

        # Sắp xếp ưu tiên các biến Directional nếu có phiếu cao
        consensus_features = sorted(
            consensus_features,
            key=lambda x: (
                feature_votes[x],
                any(k in x for k in self.directional_priority_keywords)
            ),
            reverse=True
        )

        # Fallback: Đảm bảo số lượng biến tối thiểu nếu ngưỡng quá gắt
        if len(consensus_features) < self.min_features:
            sorted_all = sorted(feature_votes.items(), key=lambda x: x[1], reverse=True)
            consensus_features = [f for f, _ in sorted_all[:self.min_features]]

        # Giới hạn số lượng biến tối đa để chống overfitting
        if len(consensus_features) > self.max_features:
            consensus_features = consensus_features[:self.max_features]

        # 4. Xác định độ trễ đồng thuận (Lấy median của các block, mặc định 1)
        final_lags = {}
        for f in consensus_features:
            if feature_lags_collected[f]:
                final_lags[f] = int(np.median(feature_lags_collected[f]))
            else:
                final_lags[f] = 1

        print(f"   ✓ Consensus thành công: {len(consensus_features)} features vượt ngưỡng ({vote_threshold}/{total_valid_blocks} blocks).")

        # 5. Biến đổi trễ & động học trên toàn bộ Train Fold dựa trên biến và lag đã đồng thuận
        X_transformed = Layer14LagTransformEngine.apply_volatility_scaled_lags(
            X_mat[consensus_features], final_lags
        )

        return X_transformed, final_lags, list(X_transformed.columns)

## Feature Engineering Packaging

In [ ]:
class Stage2DiagnosticEngine:
    """Đóng gói Stage 2: Chỉ chạy trên tập Train (Single-Asset) để trích xuất payload điều khiển."""
    def fit(self, df_train: pd.DataFrame) -> dict:
        return Stage2SingleAssetDGPScanner.execute(df_train)


class StatefulStage3FeatureEngine:
    """
    Stage 3 thu gọn: Chỉ sinh các đặc trưng PHỤ THUỘC THAM SỐ (d*, GMM)
    được chỉ định từ Stage 2 Payload.
    """
    def __init__(self, payload: dict):
        self.payload = payload
        self.opt_d = payload.get('optimal_d', 1.0)
        self.gmm_model = None
        self.bull_idx = None

    def fit(self, df_train: pd.DataFrame):
        # 1. Fit tham số GMM Regime trên Train (nếu cần dùng)
        r = df_train['log_return'].fillna(0).values
        # Sử dụng vol đã có sẵn từ ngoài fold
        vol_col = 'vol_parkinson' if 'vol_parkinson' in df_train.columns else 'log_return'
        vol = df_train[vol_col].fillna(0).values

        train_mat = np.column_stack([r, vol])
        try:
            if np.var(train_mat[:, 0]) > 1e-8:
                self.gmm_model = GaussianMixture(n_components=2, random_state=42, n_init=1)
                self.gmm_model.fit(train_mat)
                self.bull_idx = np.argmax(self.gmm_model.means_[:, 0])
        except Exception:
            self.gmm_model = None
        return self

    def transform(self, df: pd.DataFrame) -> pd.DataFrame:
        # Nhận vào DataFrame đã có sẵn đặc trưng phi tham số
        feat = df.copy()

        # 1. Bổ sung Fractional Memory với d* đã đóng băng từ Train
        if 0.0 < self.opt_d < 1.0:
            frac_strat = FractionalMemoryStrategy(optimal_d=self.opt_d)
            feat = pd.concat([feat, frac_strat.construct(df)], axis=1)

        # 2. Bổ sung GMM Regime Probabilities (chỉ predict_proba, không fit lại)
        if self.gmm_model is not None:
            r = df['log_return'].fillna(0).values
            vol_col = 'vol_parkinson' if 'vol_parkinson' in df.columns else 'log_return'
            vol = df[vol_col].fillna(0).values

            probs = self.gmm_model.predict_proba(np.column_stack([r, vol]))
            feat['gmm_prob_bull'] = probs[:, self.bull_idx]
            feat['gmm_prob_bear'] = probs[:, 1 - self.bull_idx]

        return feat


class Stage4SelectionEngine:
    """Đóng gói Stage 4: Chọn lọc đồng thuận (Consensus) trên môi trường Train Fold."""
    def __init__(self, payload: dict):
        self.payload = payload
        # Kích hoạt Router Consensus Selection
        self.router = Stage4ConsensusSelectionRouter(
            payload,
            n_blocks=8,            # 8 cửa sổ khối thời gian trượt
            block_ratio=0.75,      # Mỗi khối dài 75% tập Train
            consensus_thresh=0.60  # Yêu cầu xuất hiện trong ít nhất 60% số khối
        )
        self.optimal_lags_ = None
        self.selected_features_ = None

    def fit(self, X_train: pd.DataFrame, y_train: pd.Series):
        _, optimal_lags, final_features = self.router.execute(X_train, y_train)
        self.optimal_lags_ = optimal_lags
        self.selected_features_ = final_features
        return self

    def transform(self, X: pd.DataFrame) -> pd.DataFrame:
        if self.optimal_lags_ is None or self.selected_features_ is None:
            raise ValueError("Mô hình chưa được fit(). Vui lòng gọi fit() trên tập train trước.")

        # Áp dụng cấu hình lag và features đã được consensus phê duyệt
        X_transformed = Layer14LagTransformEngine.apply_volatility_scaled_lags(X, self.optimal_lags_)

        if 'regime_p_high_vol' in X.columns and 'regime_p_high_vol' in self.selected_features_:
            X_transformed['regime_p_high_vol'] = X['regime_p_high_vol']

        available_cols = [c for c in self.selected_features_ if c in X_transformed.columns]
        return X_transformed[available_cols]

class EconometricsFeaturePipeline(BaseEstimator, TransformerMixin):
    """
    Pipeline hoàn chỉnh trong từng Fold:
    Input: df đã enriched đặc trưng phi tham số từ ngoài fold.
    Output: Ma trận X_train/X_test tinh gọn, trực giao, không rò rỉ.
    """
    def __init__(self, target_horizon: int = 5):
        self.target_horizon = target_horizon
        self.is_fitted = False
        self.s2_engine = None
        self.s3_engine = None
        self.s4_engine = None
        self.routing_payload_ = None
        self.final_features_list_ = None

    def _prepare_target(self, df: pd.DataFrame) -> tuple[pd.DataFrame, pd.Series]:
        # 1. Tách chuỗi Nhãn
        y = df['target_label'].copy().rename("target")
        valid_idx = y.dropna().index

        # 2. KHU CÁCH LY (QUARANTINE ZONE): Danh sách chính xác các biến thuộc về tương lai/định cỡ nhãn
        quarantine_cols = [
            'target_label',          # Rò rỉ nhãn trực tiếp
            'target_barrier_return', # Tỷ suất sinh lời ở tương lai khi chạm rào
            'barrier_touch_time',    # Thời gian tương lai chạm rào
            'local_volatility',      # Biến định cỡ rào cản (bị nhiễm .bfill)
            'split_factor',          # Tham số nội suy giá trị tương lai
            'dividend'               # Tham số nội suy giá trị tương lai
        ]

        # 3. Làm sạch ma trận X
        X_clean = df.loc[valid_idx].drop(columns=quarantine_cols, errors='ignore')

        return X_clean, y.loc[valid_idx]

    def fit(self, X: pd.DataFrame, y=None):
        start_date = str(X.index.min()).split()[0]
        end_date = str(X.index.max()).split()[0]
        print(f"--- BẮT ĐẦU FIT PIPELINE TRÊN TẬP TRAIN ({start_date} đến {end_date}) ---")

        # 1. Căn chỉnh Target
        X_data, y_data = self._prepare_target(X)
        aligned = pd.concat([X_data, y_data], axis=1).dropna()
        X_train_aligned = aligned.drop(columns=['target'])
        y_train_aligned = aligned['target']

        # 2. Fit Stage 2: Diagnostic Engine để lấy d*, dynamic lags, flags
        self.s2_engine = Stage2DiagnosticEngine()
        self.routing_payload_ = self.s2_engine.fit(X_train_aligned)

        # 3. Fit & Transform Stage 3 Stateful (Học d*, fit GMM trên Train)
        self.s3_engine = StatefulStage3FeatureEngine(self.routing_payload_)
        self.s3_engine.fit(X_train_aligned)
        X_train_candidates = self.s3_engine.transform(X_train_aligned)

        # Dropna trung gian trước khi nạp vào lọc thống kê
        aligned_s3 = pd.concat([X_train_candidates, y_train_aligned], axis=1).dropna()
        X_train_candidates = aligned_s3.drop(columns=['target'])
        y_train_candidates = aligned_s3['target']

        # 4. Fit Stage 4: Feature Selection & Lag Transform
        self.s4_engine = Stage4SelectionEngine(self.routing_payload_)
        self.s4_engine.fit(X_train_candidates, y_train_candidates)

        self.final_features_list_ = self.s4_engine.selected_features_
        self.is_fitted = True

        print("--- FIT PIPELINE HOÀN TẤT ---")
        return self

    def transform(self, X: pd.DataFrame) -> pd.DataFrame:
        if not self.is_fitted:
            raise ValueError("Pipeline chưa được fit. Hãy gọi fit(df_train) trước.")

        # Sinh tiếp các feature stateful dựa trên tham số Train (d*, GMM predict)
        X_test_candidates = self.s3_engine.transform(X)

        # Biến đổi trễ động học và cắt tính năng bằng Stage 4
        X_test_final = self.s4_engine.transform(X_test_candidates)

        return X_test_final

    def fit_transform(self, X: pd.DataFrame, y=None) -> pd.DataFrame:
        return self.fit(X, y).transform(X)

# Modeling

## Data Preparation

In [ ]:
df_port = pd.read_csv(portfolio_path)
for i in range(2018, 2023):
    df = df_port.loc[df_port.year == 2023].copy()
    df.year = i
    df_port = pd.concat([df_port, df], axis="rows").sort_values(by="year")

with open("clean_price_dict.pkl", "wb") as f:
    pickle.dump(clean_price_dict, f)

df_raw = pd.DataFrame()
for year in df_port.year.unique().tolist():
    df = pd.DataFrame() # Initialize df for each year
    for tick in df_port.Ticker.unique().tolist():

        # Ensure 'weight' is a scalar. Use .item() to get the scalar value.
        # If .loc[] might return an empty series, this would raise a ValueError. Added try-except.
        try:
            weight = df_port.loc[(df_port.year == year) & (df_port.Ticker == tick), 'Weight_Final_Constrained'].item()
        except ValueError:
            print(f"Warning: No or multiple weights found for {tick} in {year}. Skipping.")
            continue

        # Use 'tick' as the key for clean_price_dict, not an undefined 'key'
        if tick in clean_price_dict:
            value = clean_price_dict[tick].copy()
            # Filter value DataFrame for the current year
            value = value.loc[value.index.year == year, :]

            # Perform the multiplication. 'weight' is now a scalar.
            if not value.empty:
                if df.empty:
                    df = weight * value
                else:
                    df += weight * value
            else:
                print(f"Warning: No data for ticker {tick} in year {year}. Skipping contribution.")
        else:
            print(f"Warning: Ticker {tick} not found in clean_price_dict. Skipping.")

    # After iterating through all tickers for a year, concatenate the yearly aggregated df to df_raw
    if not df.empty:
        df_raw = pd.concat([df_raw, df], axis="rows")
df_raw.to_csv("df_raw.csv")

# =====================================================================
# MODELING: SINGLE SYNTHETIC ASSET EXECUTION (GIỮ NGUYÊN CODE CỦA BẠN)
# =====================================================================
df = pd.read_csv("df_raw.csv")

# Chuyển đổi cột 'time' sang datetime và set làm Index (Single Index)
df['time'] = pd.to_datetime(df['time'])
df.set_index('time', inplace=True)
df.sort_index(inplace=True)

# 1. Thực thi Stage 1 Pipeline với cấu trúc Đơn tài sản (Single-Asset)
df_clean, audit_stats = Layer1DataIntegrity.run_audit(df)

# 2. Khai phá Topology Tiêu chuẩn & L2 Hooks
df_topo = Layer2MicrostructureTopologies.compute(df_clean)

print("=== EMPIRICAL AUDIT METRICS (LAYER 1) ===")
for k, v in audit_stats.items():
    print(f"- {k}: {v}")

# =====================================================================
# BƯỚC CHUẨN HÓA MỚI: TÍNH TOÁN NGOÀI FOLD (STATELESS & TARGET)
# =====================================================================
target_horizon = 5

# A. Gán nhãn Triple Barrier 1 lần duy nhất trên toàn chuỗi
# (Bảo toàn tính liên tục của chuỗi t -> t+h, triệt tiêu méo biên khi chia Fold.
# Leakage giữa Train và Test sẽ do purge_gap >= target_horizon trong PurgedCV xử lý)
print("\n[+] Đang tính toán Triple Barrier Target trên toàn bộ chuỗi...")
df_labeled = Layer0TripleBarrier.label(
    df_topo, h=target_horizon, pt=1.0, sl=1.0, vol_span=20
)

# B. Thực thi Stateless Feature Router một lần duy nhất (ngoài Fold)
# Chứa các đặc trưng chỉ trượt theo quá khứ: Moving Average, CMF, Parkinson Vol, Kinematics...
print("[+] Đang sinh các đặc trưng phi tham số (Stateless Features)...")
stateless_router = StatelessFeatureRouter(default_lags=[1, 3, 5, 10, 20])
df_features_stateless = stateless_router.execute(df_labeled)

# Ghép nhãn mục tiêu và tập đặc trưng phi tham số vào dataset hoàn chỉnh
df_final = df_features_stateless.copy()
df_final.to_csv("df_final.csv")
print(f"[+] DATASET SẴN SÀNG CHO PHÂN CHIA CV: {len(df_final)} phiên với {df_final.shape[1]} cột ban đầu.")

=== EMPIRICAL AUDIT METRICS (LAYER 1) ===
- initial_bars: 1927
- clean_bars: 1295
- dropped_flat_cols: []

[+] Đang tính toán Triple Barrier Target trên toàn bộ chuỗi...
[+] Đang sinh các đặc trưng phi tham số (Stateless Features)...
[+] DATASET SẴN SÀNG CHO PHÂN CHIA CV: 1295 phiên với 107 cột ban đầu.


In [ ]:
pd.read_csv("df_final.csv").isna().sum().sum()

np.int64(1415)

## Model & Back-testing

In [ ]:
# =====================================================================
# 1. PURGED TIME-SERIES CV (ZERO LEAKAGE)
# =====================================================================
class StandardPurgedCV:

  def __init__(self, n_splits=5, purge_gap=5, max_train_size=None):
    self.n_splits = n_splits
    self.purge_gap = purge_gap
    self.max_train_size = max_train_size

  def split(self, X, y=None, groups=None):
    tscv = TimeSeriesSplit(
        n_splits=self.n_splits, max_train_size=self.max_train_size
    )
    for tr_idx, te_idx in tscv.split(X):
      tr_purged = tr_idx[tr_idx < (te_idx[0] - self.purge_gap)]
      if len(tr_purged) > 0 and len(te_idx) > 0:
        yield tr_purged, te_idx

  def get_n_splits(self, X=None, y=None, groups=None):
    return self.n_splits

### Thực Thi Modeling Pipeline

In [ ]:
n_outer_splits = 5
buffer_days = 60
MAX_TRAIN_SIZE = 750
THRESHOLD = 0.50

outer_cv = StandardPurgedCV(
    n_splits=n_outer_splits,
    purge_gap=target_horizon,   # Triệt tiêu hoàn toàn rò rỉ nhãn giữa Train và Test
    max_train_size=MAX_TRAIN_SIZE,
)

oos_predictions = []
trained_models = []
selected_features_per_fold = []

print("\n" + "=" * 80)
print("SINGLE LOGISTIC REGRESSION: PURGED CV & STATEFUL PIPELINE")
print("=" * 80)

for fold, (train_idx, test_idx) in enumerate(outer_cv.split(df_final)):
    print(f"\n>>> [FOLD {fold + 1}/{n_outer_splits}]")

    # 1. Trích xuất dữ liệu Train & Test từ DataFrame đã làm giàu sẵn
    df_train = df_final.iloc[train_idx].copy()
    df_test = df_final.iloc[test_idx].copy()

    # 2. Loại bỏ cột Zero-Variance cục bộ trên tập Train
    numeric_cols = df_train.select_dtypes(include=[np.number]).columns
    zero_cols = [c for c in numeric_cols if df_train[c].std(ddof=0) < 1e-8]
    if zero_cols:
        df_train.drop(columns=zero_cols, inplace=True)
        df_test.drop(columns=zero_cols, inplace=True, errors="ignore")

    # 3. Khởi tạo và Fit Stateful Pipeline (Học d*, fit GMM, chọn lọc VIF/HRP/Granger TRÊN TRAIN)
    pipeline = EconometricsFeaturePipeline(target_horizon=target_horizon)
    pipeline.fit(df_train)

    current_features = list(pipeline.final_features_list_)
    selected_features_per_fold.append(current_features)

    # Transform trên tập Train để chuẩn bị ma trận huấn luyện
    X_train_transformed = pipeline.transform(df_train)
    _, y_train_series = pipeline._prepare_target(df_train)

    common_train_idx = X_train_transformed.index.intersection(y_train_series.index)
    X_tr = X_train_transformed.loc[common_train_idx, current_features]
    y_tr = y_train_series.loc[common_train_idx].astype(int).values

    print(
        f"    [Train Samples] Số phiên huấn luyện: {len(X_tr)} (Lớp 1: "
        f"{np.sum(y_tr == 1)}, Lớp 0: {np.sum(y_tr == 0)})")

    if len(np.unique(y_tr)) < 2:
        print(f"    [!] Fold {fold + 1} chỉ có 1 class duy nhất. Bỏ qua fold này.")
        continue

    # 4. Pipeline Estimator: Impute median (từ Train) -> Scaler (từ Train) -> Logistic Regression
    model = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        (
            "clf",
            LogisticRegression(
                penalty="l2",
                C=0.1,
                class_weight="balanced",
                solver="lbfgs",
                max_iter=1000,
                random_state=42,
            ),
        ),
    ])

    model.fit(X_tr, y_tr)
    trained_models.append(model)

    # 5. Suy luận Out-Of-Sample (OOS) trên Test Set
    # Bổ sung buffer phía trước Test để các biến độ trễ (Lag) và FracDiff tính toán liền mạch
    buffer_start = max(0, test_idx[0] - buffer_days)
    df_test_buffer = df_final.iloc[buffer_start : test_idx[-1] + 1].copy()

    # Transform bằng các tham số đã đóng băng (d*, GMM predict, Selection)
    X_test_transformed = pipeline.transform(df_test_buffer)

    # Cắt lọc chính xác vùng thời gian thuộc Test Set
    X_test_pure = X_test_transformed.loc[X_test_transformed.index.isin(df_test.index)]
    _, y_test_series = pipeline._prepare_target(df_test)

    common_test_idx = X_test_pure.index.intersection(y_test_series.index)
    X_te = X_test_pure.loc[common_test_idx].reindex(
        columns=current_features, fill_value=np.nan
    )
    y_te = y_test_series.loc[common_test_idx].astype(int).values

    # Dự báo xác suất OOS
    raw_test_probs = model.predict_proba(X_te)[:, 1]
    y_pred_binary = (raw_test_probs >= THRESHOLD).astype(int)

    # Định cỡ vị thế tuyến tính (Position Sizing)
    bet_sizes = np.where(
        raw_test_probs >= THRESHOLD,
        np.clip((raw_test_probs - THRESHOLD) / (1.0 - THRESHOLD), 0.2, 1.0),
        0.0,
    )

    res_df = pd.DataFrame(
        {
            "fold": fold + 1,
            "prob_up": raw_test_probs,
            "pred_signal": y_pred_binary,
            "bet_size": bet_sizes,
            "true_target": y_te,
        },
        index=common_test_idx,
    )

    oos_predictions.append(res_df)

print("\n[✓] HOÀN TẤT TOÀN BỘ PURGED CV.")



SINGLE LOGISTIC REGRESSION: PURGED CV & STATEFUL PIPELINE

>>> [FOLD 1/5]
--- BẮT ĐẦU FIT PIPELINE TRÊN TẬP TRAIN (2018-08-03 đến 2019-05-30) ---
   -> Bắt đầu Consensus Selection qua 8 Moving Blocks (Kích thước block: 72/96 phiên)...
   ✓ Consensus thành công: 7 features vượt ngưỡng (5/8 blocks).
--- FIT PIPELINE HOÀN TẤT ---
    [Train Samples] Số phiên huấn luyện: 215 (Lớp 1: 117, Lớp 0: 98)

>>> [FOLD 2/5]
--- BẮT ĐẦU FIT PIPELINE TRÊN TẬP TRAIN (2018-08-03 đến 2020-10-26) ---
   -> Bắt đầu Consensus Selection qua 8 Moving Blocks (Kích thước block: 196/262 phiên)...
   ✓ Consensus thành công: 9 features vượt ngưỡng (5/8 blocks).
--- FIT PIPELINE HOÀN TẤT ---
    [Train Samples] Số phiên huấn luyện: 430 (Lớp 1: 218, Lớp 0: 212)

>>> [FOLD 3/5]
--- BẮT ĐẦU FIT PIPELINE TRÊN TẬP TRAIN (2018-08-03 đến 2021-08-23) ---
   -> Bắt đầu Consensus Selection qua 8 Moving Blocks (Kích thước block: 263/351 phiên)...
   ✓ Consensus thành công: 11 features vượt ngưỡng (5/8 blocks).
--- FIT PIPELI

### Phân Phối Tín Hiệu (Signal Distribution)

In [ ]:
# =====================================================================
# 3. ĐÁNH GIÁ METRICS DỰ BÁO SINGLE LOGISTIC REGRESSION (OOS)
# =====================================================================
print("\n" + "=" * 65)
print("       PHÂN PHỐI TÍN HIỆU DỰ BÁO OOS THEO FOLD (LOGISTIC)")
print("=" * 65)

# 1. Map nhãn nhị phân: 0 (Đứng ngoài / Tiền mặt) và 1 (Vào lệnh Mua)
binary_label_map = {0: 'No Trade (0)', 1: 'Enter Long (1)'}
df_oos_results = pd.concat(oos_predictions).sort_index()
df_oos_results['signal_label'] = df_oos_results['pred_signal'].map(
    binary_label_map
)

fold_pct = (
    pd.crosstab(
        df_oos_results['fold'],
        df_oos_results['signal_label'],
        normalize='index',
    )
    * 100
)
print(fold_pct.round(2).to_string())

# 2. Thống kê Bet Sizing theo từng Fold
print("\n" + "-" * 65)
print("    THỐNG KÊ QUY MÔ VỊ THẾ (BET SIZE DISTRIBUTION)")
print("-" * 65)
sizing_summary = df_oos_results.groupby('fold')['bet_size'].agg(
    Active_Trade_Pct=lambda x: (x > 0).mean() * 100,
    Mean_Position='mean',
    Max_Position='max',
)
print(sizing_summary.round(2).to_string())

# 3. Báo cáo phân loại nhị phân
print("\n" + "=" * 65)
print("     BÁO CÁO HIỆU NĂNG PHÂN LOẠI LOGISTIC REGRESSION (OOS)")
print("=" * 65)

y_true = df_oos_results['true_target'].astype(int)
y_pred = df_oos_results['pred_signal'].astype(int)
y_prob = df_oos_results['prob_up']

print(
    classification_report(
        y_true,
        y_pred,
        target_names=['Fail / Time-out (0)', 'Take-Profit Hit (1)'],
        digits=4,
    )
)

# 4. Các chỉ số chất lượng xác suất (Probability Quality Metrics)
roc_auc = roc_auc_score(y_true, y_prob)
brier = brier_score_loss(y_true, y_prob)
print(f"ROC-AUC Score (Khả năng phân tách xác suất) : {roc_auc:.4f}")
print(f"Brier Score Loss (Độ hiệu chuẩn xác suất)    : {brier:.4f}")

# 5. Ma trận nhầm lẫn chuẩn 2x2
print("\nMa trận nhầm lẫn (Confusion Matrix):")
cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
cm_df = pd.DataFrame(
    cm,
    index=['Thực tế Thua/Hòa (0)', 'Thực tế Thắng (1)'],
    columns=['Dự đoán Bỏ qua (0)', 'Dự đoán Mua (1)'],
)
print(cm_df)

# 6. Kiểm tra tính ổn định đặc trưng xuyên suốt các Folds
if selected_features_per_fold:
  # Corrected: Convert each sub-list to a set before performing the union
  all_feats = set().union(*[set(s) for s in selected_features_per_fold])
  consensus_counts = {
      f: sum(f in fold_set for fold_set in selected_features_per_fold)
      for f in all_feats
  }
  df_consensus = (
      pd.Series(consensus_counts)
      .sort_values(ascending=False)
      .to_frame(name='Fold_Appearance_Count')
  )

  print("\n" + "=" * 65)
  print("    TÍNH ỔN ĐỊNH CỦA ĐẶC TRƯNG XUYÊN SUỐT CÁC FOLDS (CONSENSUS)")
  print("=" * 65)
  print(df_consensus.head(15))


       PHÂN PHỐI TÍN HIỆU DỰ BÁO OOS THEO FOLD (LOGISTIC)
signal_label  Enter Long (1)  No Trade (0)
fold                                      
1                      37.67         62.33
2                      53.02         46.98
3                      49.77         50.23
4                      46.51         53.49
5                      49.05         50.95

-----------------------------------------------------------------
    THỐNG KÊ QUY MÔ VỊ THẾ (BET SIZE DISTRIBUTION)
-----------------------------------------------------------------
      Active_Trade_Pct  Mean_Position  Max_Position
fold                                               
1                37.67           0.09          0.97
2                53.02           0.15          1.00
3                49.77           0.11          0.59
4                46.51           0.10          0.41
5                49.05           0.10          0.44

     BÁO CÁO HIỆU NĂNG PHÂN LOẠI LOGISTIC REGRESSION (OOS)
                     precision  

# Dianogsis Test:

## Test 1: Đo Lường Bias - Variance (Train vs. Out-of-Sample Test Gap)

In [ ]:
# ---------------------------------------------------------------------
# Mục tiêu: Xác định chính xác mô hình đang bị High Bias (Underfitting)
# hay High Variance (Overfitting).
# ---------------------------------------------------------------------
print("=" * 80)
print("TEST 1: PHÂN TÍCH LỆCH BẬC BIAS - VARIANCE (TRAIN METRICS vs OOS METRICS)")
print("=" * 80)

train_metrics = []
test_metrics = []

for fold, (train_idx, test_idx) in enumerate(outer_cv.split(df_final)):
    model = trained_models[fold]
    cur_feats = selected_features_per_fold[fold]

    # 1. Tái hiện tập Train cho Fold này
    df_train = df_final.iloc[train_idx].copy()
    pipeline = EconometricsFeaturePipeline(target_horizon=target_horizon)
    pipeline.fit(df_train)

    X_tr_trans = pipeline.transform(df_train)
    _, y_tr_series = pipeline._prepare_target(df_train)
    common_tr = X_tr_trans.index.intersection(y_tr_series.index)
    X_tr = X_tr_trans.loc[common_tr, cur_feats]
    y_tr = y_tr_series.loc[common_tr].astype(int).values

    # Train Metrics
    p_tr = model.predict_proba(X_tr)[:, 1]
    y_pred_tr = (p_tr >= THRESHOLD).astype(int)

    train_metrics.append({
        "fold": fold + 1,
        "train_acc": accuracy_score(y_tr, y_pred_tr),
        "train_auc": roc_auc_score(y_tr, p_tr),
        "train_brier": brier_score_loss(y_tr, p_tr),
        "train_loss": log_loss(y_tr, p_tr),
    })

    # OOS Test Metrics của Fold tương ứng
    res_fold = oos_predictions[fold]
    test_metrics.append({
        "fold": fold + 1,
        "test_acc": accuracy_score(res_fold["true_target"], res_fold["pred_signal"]),
        "test_auc": roc_auc_score(res_fold["true_target"], res_fold["prob_up"]),
        "test_brier": brier_score_loss(res_fold["true_target"], res_fold["prob_up"]),
        "test_loss": log_loss(res_fold["true_target"], res_fold["prob_up"]),
    })

df_diag = pd.DataFrame(train_metrics).merge(pd.DataFrame(test_metrics), on="fold")
df_diag["acc_gap (Tr - Te)"] = df_diag["train_acc"] - df_diag["test_acc"]
df_diag["auc_gap (Tr - Te)"] = df_diag["train_auc"] - df_diag["test_auc"]

print(df_diag[[
    "fold", "train_acc", "test_acc", "acc_gap (Tr - Te)",
    "train_auc", "test_auc", "auc_gap (Tr - Te)"
]].to_string(index=False))

print("-" * 80)
print(f"[*] Train AUC trung bình : {df_diag['train_auc'].mean():.4f} | OOS Test AUC trung bình : {df_diag['test_auc'].mean():.4f}")
print(f"[*] Train ACC trung bình : {df_diag['train_acc'].mean():.4f} | OOS Test ACC trung bình : {df_diag['test_acc'].mean():.4f}")

TEST 1: PHÂN TÍCH LỆCH BẬC BIAS - VARIANCE (TRAIN METRICS vs OOS METRICS)
--- BẮT ĐẦU FIT PIPELINE TRÊN TẬP TRAIN (2018-08-03 đến 2019-05-30) ---
   -> Bắt đầu Consensus Selection qua 8 Moving Blocks (Kích thước block: 72/96 phiên)...
   ✓ Consensus thành công: 7 features vượt ngưỡng (5/8 blocks).
--- FIT PIPELINE HOÀN TẤT ---
--- BẮT ĐẦU FIT PIPELINE TRÊN TẬP TRAIN (2018-08-03 đến 2020-10-26) ---
   -> Bắt đầu Consensus Selection qua 8 Moving Blocks (Kích thước block: 196/262 phiên)...
   ✓ Consensus thành công: 9 features vượt ngưỡng (5/8 blocks).
--- FIT PIPELINE HOÀN TẤT ---
--- BẮT ĐẦU FIT PIPELINE TRÊN TẬP TRAIN (2018-08-03 đến 2021-08-23) ---
   -> Bắt đầu Consensus Selection qua 8 Moving Blocks (Kích thước block: 263/351 phiên)...
   ✓ Consensus thành công: 11 features vượt ngưỡng (5/8 blocks).
--- FIT PIPELINE HOÀN TẤT ---
--- BẮT ĐẦU FIT PIPELINE TRÊN TẬP TRAIN (2019-01-11 đến 2022-06-20) ---
   -> Bắt đầu Consensus Selection qua 8 Moving Blocks (Kích thước block: 261/349 phi

## Test 2: Kiểm Định Phi Tuyến Tính & Hiệu Ứng Tương Tác (Interaction Test)


In [ ]:
# ---------------------------------------------------------------------
# Mục tiêu: Kiểm tra xem việc bổ sung tương tác bậc hai (X_i * X_j) có phá vỡ
# giới hạn tuyến tính (Linear Bottleneck) của Logistic hay không.
# ---------------------------------------------------------------------
print("\n" + "=" * 80)
print("TEST 2: KIỂM ĐỊNH GIẢ THUYẾT PHI TUYẾN TÍNH (INTERACTION DEGREE-2 TEST)")
print("=" * 80)

interaction_aucs = []
base_aucs = []

for fold, (train_idx, test_idx) in enumerate(outer_cv.split(df_final)):
    cur_feats = selected_features_per_fold[fold]

    # Chuẩn bị Train
    df_train = df_final.iloc[train_idx].copy()
    pipeline = EconometricsFeaturePipeline(target_horizon=target_horizon)
    pipeline.fit(df_train)
    X_tr_trans = pipeline.transform(df_train)
    _, y_tr_series = pipeline._prepare_target(df_train)
    c_tr = X_tr_trans.index.intersection(y_tr_series.index)
    X_tr = X_tr_trans.loc[c_tr, cur_feats]
    y_tr = y_tr_series.loc[c_tr].astype(int).values

    # Chuẩn bị Test
    buffer_start = max(0, test_idx[0] - buffer_days)
    df_test_buffer = df_final.iloc[buffer_start : test_idx[-1] + 1].copy()
    X_te_trans = pipeline.transform(df_test_buffer)
    X_te_pure = X_te_trans.loc[X_te_trans.index.isin(df_final.iloc[test_idx].index)]
    _, y_te_series = pipeline._prepare_target(df_final.iloc[test_idx])
    c_te = X_te_pure.index.intersection(y_te_series.index)
    X_te = X_te_pure.loc[c_te, cur_feats]
    y_te = y_te_series.loc[c_te].astype(int).values

    # Impute missing values before PolynomialFeatures
    imputer = SimpleImputer(strategy="median")
    X_tr_imputed = imputer.fit_transform(X_tr)
    X_te_imputed = imputer.transform(X_te)

    # Tạo biến tương tác thuần túy X_i * X_j
    poly = PolynomialFeatures(degree=2, interaction_only=True, include_bias=False)
    X_tr_poly = poly.fit_transform(X_tr_imputed)
    X_te_poly = poly.transform(X_te_imputed)

    # Huấn luyện mô hình Logistic có tương tác phi tuyến (tăng nhẹ L2 regularization C=0.05)
    model_poly = Pipeline([
        # Imputer is already applied above
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(penalty="l2", C=0.05, class_weight="balanced", solver="lbfgs", max_iter=1000, random_state=42))
    ])
    model_poly.fit(X_tr_poly, y_tr)
    p_poly = model_poly.predict_proba(X_te_poly)[:, 1]

    # So sánh với mô hình gốc
    # Ensure the base model also works on imputed data if it expects it, or original if it handles NaNs itself.
    # For consistency, assume the original `trained_models` were trained on data that eventually got imputed by their internal pipeline.
    # For this comparison, we'll re-run the base model's prediction using the imputed X_te to ensure fair comparison with X_te_poly
    # Or, if `trained_models[fold]` expects un-imputed data, it should be passed X_te directly. Reverting X_te_imputed for p_base to match behavior if `trained_models` handles imputation internally.
    # If the `trained_models` pipeline has an imputer, X_te can be passed directly.
    p_base = trained_models[fold].predict_proba(X_te)[:, 1]

    base_aucs.append(roc_auc_score(y_te, p_base))
    interaction_aucs.append(roc_auc_score(y_te, p_poly))

df_poly_res = pd.DataFrame({
    "Fold": range(1, len(base_aucs) + 1),
    "Linear Baseline AUC": base_aucs,
    "Interaction (Deg 2) AUC": interaction_aucs
})
df_poly_res["Delta AUC"] = df_poly_res["Interaction (Deg 2) AUC"] - df_poly_res["Linear Baseline AUC"]
print(df_poly_res.to_string(index=False))
print(f"[*] Delta AUC trung bình khi bổ sung phi tuyến: {df_poly_res['Delta AUC'].mean():+.4f}")


TEST 2: KIỂM ĐỊNH GIẢ THUYẾT PHI TUYẾN TÍNH (INTERACTION DEGREE-2 TEST)
--- BẮT ĐẦU FIT PIPELINE TRÊN TẬP TRAIN (2018-08-03 đến 2019-05-30) ---
   -> Bắt đầu Consensus Selection qua 8 Moving Blocks (Kích thước block: 72/96 phiên)...
   ✓ Consensus thành công: 7 features vượt ngưỡng (5/8 blocks).
--- FIT PIPELINE HOÀN TẤT ---
--- BẮT ĐẦU FIT PIPELINE TRÊN TẬP TRAIN (2018-08-03 đến 2020-10-26) ---
   -> Bắt đầu Consensus Selection qua 8 Moving Blocks (Kích thước block: 196/262 phiên)...
   ✓ Consensus thành công: 9 features vượt ngưỡng (5/8 blocks).
--- FIT PIPELINE HOÀN TẤT ---
--- BẮT ĐẦU FIT PIPELINE TRÊN TẬP TRAIN (2018-08-03 đến 2021-08-23) ---
   -> Bắt đầu Consensus Selection qua 8 Moving Blocks (Kích thước block: 263/351 phiên)...
   ✓ Consensus thành công: 11 features vượt ngưỡng (5/8 blocks).
--- FIT PIPELINE HOÀN TẤT ---
--- BẮT ĐẦU FIT PIPELINE TRÊN TẬP TRAIN (2019-01-11 đến 2022-06-20) ---
   -> Bắt đầu Consensus Selection qua 8 Moving Blocks (Kích thước block: 261/349 phiê

## Test 3: Phân Rã Brier Score & Đường Cong Hiệu Chuẩn (Calibration Diagnosis)

In [ ]:
# ---------------------------------------------------------------------
# Mục tiêu: Đánh giá độ tin cậy của xác suất dự báo (Reliability vs Resolution).
# ---------------------------------------------------------------------
print("\n" + "=" * 80)
print("TEST 3: PHÂN RÃ BRIER SCORE & HIỆU CHUẨN XÁC SUẤT (CALIBRATION ANALYSIS)")
print("=" * 80)

df_all_oos = pd.concat(oos_predictions).sort_index()
y_true_all = df_all_oos["true_target"].astype(int).values
p_pred_all = df_all_oos["prob_up"].values

# Phân rã Brier theo Murphy (1973)
base_rate = np.mean(y_true_all)
uncertainty = base_rate * (1 - base_rate)
actual_brier = brier_score_loss(y_true_all, p_pred_all)

# Chia thành 5 bins xác suất
prob_true, prob_pred = calibration_curve(y_true_all, p_pred_all, n_bins=5, strategy="uniform")

print(f"1. Tỷ lệ lớp dương thực tế (Base Rate)  : {base_rate:.4f}")
print(f"2. Sai số ngẫu nhiên tối thiểu (Uncertainty): {uncertainty:.4f}")
print(f"3. Brier Score thực tế của Logistic         : {actual_brier:.4f}")
if actual_brier > uncertainty:
    print("   -> KẾT LUẬN: Brier Score > Uncertainty. Xác suất của mô hình làm tăng thêm sai số so với việc đoán xác suất cố định!")
else:
    print("   -> KẾT LUẬN: Mô hình có đóng góp giá trị thông tin xác suất (Resolution > Reliability penalty).")

calib_table = pd.DataFrame({
    "Dải Xác Suất Dự Báo": [f"{p:.2f}" for p in prob_pred],
    "Tỷ Lệ Thực Tế Thắng (Empirical Winrate)": [f"{t:.2f}" for t in prob_true],
    "Sai Lệch (Calibration Bias)": [f"{(p - t):+.2f}" for p, t in zip(prob_pred, prob_true)]
})
print("\nChi tiết đường cong hiệu chuẩn (Binned Probabilities):")
print(calib_table.to_string(index=False))


TEST 3: PHÂN RÃ BRIER SCORE & HIỆU CHUẨN XÁC SUẤT (CALIBRATION ANALYSIS)
1. Tỷ lệ lớp dương thực tế (Base Rate)  : 0.5187
2. Sai số ngẫu nhiên tối thiểu (Uncertainty): 0.2497
3. Brier Score thực tế của Logistic         : 0.2579
   -> KẾT LUẬN: Brier Score > Uncertainty. Xác suất của mô hình làm tăng thêm sai số so với việc đoán xác suất cố định!

Chi tiết đường cong hiệu chuẩn (Binned Probabilities):
Dải Xác Suất Dự Báo Tỷ Lệ Thực Tế Thắng (Empirical Winrate) Sai Lệch (Calibration Bias)
               0.09                                    0.09                       +0.00
               0.35                                    0.53                       -0.18
               0.50                                    0.53                       -0.03
               0.65                                    0.52                       +0.14
               0.93                                    0.36                       +0.57


## Test 4: Permutation Feature Importance Trên Tập OOS Test

In [ ]:
# ---------------------------------------------------------------------
# Mục tiêu: Xác định xem tín hiệu đến từ biến nào hay toàn bộ là giả định ngẫu nhiên.
# ---------------------------------------------------------------------
print("\n" + "=" * 80)
print("TEST 4: OOS PERMUTATION TEST (KIỂM ĐỊNH TÍNH CHÂN THỰC CỦA ĐẶC TRƯNG)")
print("=" * 80)

# Chạy Permutation trên Fold cuối (Fold 5) để kiểm tra độ nhạy
fold_idx = 4 # Index for the last fold (0-indexed for lists)
model_eval = trained_models[fold_idx]
feats_eval = selected_features_per_fold[fold_idx] # Features selected for this fold

# Get train and test indices for this specific fold from outer_cv
# Note: outer_cv.split returns an iterator. We need to iterate it to the specific fold_idx.
all_folds_indices = list(outer_cv.split(df_final))
train_indices, test_indices = all_folds_indices[fold_idx]

# Prepare data for this fold
df_train_fold = df_final.iloc[train_indices].copy()
df_test_fold = df_final.iloc[test_indices].copy() # This is the original test split

# Re-initialize and fit the pipeline for this fold's training data
# This is crucial as pipeline parameters (d*, GMM, feature selection) are fold-specific
pipeline_fold_eval = EconometricsFeaturePipeline(target_horizon=target_horizon)
pipeline_fold_eval.fit(df_train_fold)

# Transform the test data, including buffer, then extract the pure test set
buffer_start = max(0, test_indices[0] - buffer_days)
df_test_buffer_fold = df_final.iloc[buffer_start : test_indices[-1] + 1].copy()
X_test_transformed_full = pipeline_fold_eval.transform(df_test_buffer_fold)

# Extract X_te and y_te for this fold, aligning indices and handling NaNs exactly as in CV loop
X_test_pure_fold = X_test_transformed_full.loc[X_test_transformed_full.index.isin(df_test_fold.index)]
_, y_test_series_fold = pipeline_fold_eval._prepare_target(df_test_fold)

common_eval_idx = X_test_pure_fold.index.intersection(y_test_series_fold.index)

X_te_raw = X_test_pure_fold.loc[common_eval_idx].reindex(columns=feats_eval)
y_te_raw = y_test_series_fold.loc[common_eval_idx].astype(int).values

baseline_auc = roc_auc_score(y_te_raw, model_eval.predict_proba(X_te_raw)[:, 1])
perm_results = {}

np.random.seed(42)
for col in feats_eval:
    X_perm = X_te_raw.copy()
    X_perm[col] = np.random.permutation(X_perm[col].values)
    perm_auc = roc_auc_score(y_te_raw, model_eval.predict_proba(X_perm)[:, 1])
    perm_results[col] = baseline_auc - perm_auc

df_perm = pd.Series(perm_results).sort_values(ascending=False).to_frame(name="OOS AUC Drop (Importance)")
print(df_perm.to_string())


TEST 4: OOS PERMUTATION TEST (KIỂM ĐỊNH TÍNH CHÂN THỰC CỦA ĐẶC TRƯNG)
--- BẮT ĐẦU FIT PIPELINE TRÊN TẬP TRAIN (2019-11-08 đến 2024-02-27) ---
   -> Bắt đầu Consensus Selection qua 8 Moving Blocks (Kích thước block: 284/379 phiên)...
   ✓ Consensus thành công: 9 features vượt ngưỡng (5/8 blocks).
--- FIT PIPELINE HOÀN TẤT ---
                                OOS AUC Drop (Importance)
Dividends_momentum                               0.026318
buying_tail_power_zscaled_lag4                   0.018931
obv_zscore_20_momentum                           0.018931
geo_upper_shadow_momentum                        0.013482
obv_zscore_20_acceleration                       0.010804
gmm_prob_bull_zscaled_lag1                       0.008865
Dividends_acceleration                           0.008034
liq_amihud_z_zscaled_lag2                        0.007111
log_return_momentum                              0.006279
liq_amihud_z_acceleration                        0.005818
vol_parkinson_zscaled_lag5        